# Paper figures
Run the setup cell once, then any individual figure cell. The next 31 cells each contain data reading and actual plot code for one complete figure. Inputs are prepared in results/ by the scientific replay.


In [ ]:

# Run this setup once, then run any individual figure cell.
from pathlib import Path
import sys
import json
import shutil
from types import SimpleNamespace
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
if not (ROOT / "src/thermal_heterogeneity").is_dir():
    raise FileNotFoundError("Open this notebook from code or code/plot.")
RESULTS, PLOT = ROOT / "results", ROOT / "plot"
INTERMEDIATE = ROOT / "output/intermediate/notebook"
for folder in (PLOT / "main", PLOT / "supplementary", INTERMEDIATE):
    folder.mkdir(parents=True, exist_ok=True)
for folder in (ROOT / "src", ROOT / "src/analysis/scripts",
               ROOT / "src/analysis/energy_service_paper/scripts"):
    sys.path.insert(0, str(folder))
from thermal_heterogeneity.figures import apply_figure_style  # noqa: E402
from utilities.prepare_plot_data import refresh_plot_inputs  # noqa: E402
apply_figure_style()

# Export helper only. Each cell below contains data selection and actual plotting.
def save_plot(fig, label, dpi=600, size_mm=None, round_pixels=True, exact_pixels=False):
    from matplotlib.text import Text
    if size_mm is not None:
        fig.set_size_inches(size_mm[0]/25.4, size_mm[1]/25.4, forward=True)
    directory = PLOT / ("supplementary" if label.startswith("Figure_S") else "main")
    stem = directory / label
    for artist in fig.findobj(Text):
        artist.set_color("black")
    for extension in ("pdf", "svg"):
        metadata = {"CreationDate":None, "ModDate":None} if extension == "pdf" else {"Date":None}
        fig.savefig(stem.with_suffix("."+extension), metadata=metadata)
    for suffix, resolution in (("", dpi), ("_word", 300)):
        path = Path(str(stem)+suffix+".png")
        original_size = tuple(fig.get_size_inches())
        if exact_pixels:
            expected = tuple(round(value * resolution) for value in original_size)
            fig.set_size_inches(expected[0]/resolution, expected[1]/resolution, forward=True)
        fig.savefig(path, dpi=resolution)
        with Image.open(path) as image:
            rgb = image.convert("RGB")
            size = tuple(round(value * resolution) for value in fig.get_size_inches())
            if round_pixels and rgb.size != size:
                rgb = rgb.resize(size, Image.Resampling.LANCZOS)
            rgb.save(path)
        fig.set_size_inches(original_size, forward=True)
    caption = RESULTS / "captions" / (label+".txt")
    if caption.is_file():
        shutil.copyfile(caption, Path(str(stem)+"_caption.txt"))
    plt.close(fig)
    refresh_plot_inputs(RESULTS)
    print(label+": "+str(stem.with_suffix(".png")))
    if "get_ipython" in globals():
        from IPython.display import display, Image as NotebookImage
        display(NotebookImage(filename=str(stem.with_suffix(".png")), width=850))
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Rectangle

FONT = "Times New Roman"
CONTROL_ORDER = ("uniform_average", "personal_center", "personal_range")
CONTROL_LABELS = {
    "uniform_average": "Uniform average",
    "personal_center": "Personal centre",
    "personal_range": "Personal range",
}
CONTROL_COLORS = {
    "uniform_average": "#858684",
    "personal_center": "#C6A064",
    "personal_range": "#78A28E",
}
SOURCE_MARKERS = {"ASHRAE": "o", "CTCD": "s"}
CONTEXT_ORDER = ("Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019")
CONTEXT_LABELS = {"Miami":"Miami", "NewDelhi":"New Delhi", "PortAngeles_small2019":"Port Angeles", "Seattle_small2019":"Seattle"}

def final_style(ax, zero=False):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#777777")
    ax.spines["bottom"].set_color("#777777")
    ax.tick_params(colors="black", width=0.8, length=3)
    if zero:
        ax.axhline(0, color="#B5B5B5", lw=0.8, zorder=0)
    return ax

def final_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", fontsize=12, fontweight="bold", pad=7)

def source_control_legend(ax, location="upper center", columns=3):
    handles = [Line2D([0],[0], marker="o", linestyle="", color=CONTROL_COLORS[c],
                      label=CONTROL_LABELS[c], markersize=6) for c in CONTROL_ORDER]
    ax.legend(handles=handles, loc=location, ncol=columns, frameon=False)


In [ ]:
# Figure_01: edit this cell and rerun to update this figure.
def draw_figure_01():
    LABEL = "Figure_01"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import json

    from pathlib import Path

    from typing import Callable

    import matplotlib.pyplot as plt

    import numpy as np

    import pandas as pd

    from matplotlib.axes import Axes

    from matplotlib.figure import Figure

    from matplotlib.patches import FancyBboxPatch, Patch, Rectangle

    from thermal_heterogeneity.figures import (
        CONTEXT_COLORS as SHARED_CONTEXT_COLORS,
        DEMAND_CMAP,
        VARIANCE_COLORS,
        PALETTE,
        create_panel_figure,
        data_safe_legend,
        draw_world_land,
        style_panel,
    )

    CONTEXT_ORDER = ["Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019"]

    CONTEXT_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_LEGEND_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_COLORS = SHARED_CONTEXT_COLORS

    POPULATION_LABELS = {
        "us_archive_stratum": "United States",
        "multinational_source_balanced": "Multinational",
        "india_archive_stratum": "India",
    }

    OUTCOME_ORDER = [
        "same_direction_decrease",
        "same_direction_increase",
        "all_tied",
        "direction_not_stable",
        "match_lost",
        "all_unmatched",
    ]

    OUTCOME_LABELS = {
        "same_direction_decrease": "Decrease",
        "same_direction_increase": "Increase",
        "all_tied": "Tie",
        "direction_not_stable": "Unstable",
        "match_lost": "Unmatched",
        "all_unmatched": "Unmatched",
    }

    OUTCOME_COLORS = {
        "same_direction_decrease": PALETTE["blue"],
        "same_direction_increase": PALETTE["vermillion"],
        "all_tied": PALETTE["grey"],
        "direction_not_stable": PALETTE["purple"],
        "match_lost": PALETTE["light_grey"],
        "all_unmatched": PALETTE["light_grey"],
    }

    def _read_csv(root: Path, panel_id: str, required: set[str]) -> pd.DataFrame:
        path = root / f"{panel_id}.csv"
        frame = pd.read_csv(path)
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"{panel_id} missing columns: {sorted(missing)}")
        if frame.empty or set(frame["panel_id"]) != {panel_id}:
            raise ValueError(f"{panel_id} is empty or carries the wrong panel_id")
        return frame

    def _finish_axis(axis: Axes, letter: str, title: str, grid_axis: str | None = "y") -> None:
        style_panel(axis, letter, title, grid_axis=grid_axis)

    def build_figure_1(panel_root: Path | str) -> Figure:
        """Build Figure 1: observed geography, policy coordinates and variance scale."""

        root = Path(panel_root)
        a = _read_csv(root, "Fig1a", {"population", "records", "analysis_units"})
        b = _read_csv(
            root,
            "Fig1b",
            {"city", "latitude", "longitude", "supported", "analysis_units", "demand_center_c"},
        )
        c = _read_csv(
            root,
            "Fig1c",
            {"city", "latitude", "longitude", "supported", "analysis_units", "demand_center_c"},
        )
        d = _read_csv(
            root,
            "Fig1d",
            {"aggregation_level", "season", "name", "p10_c", "p50_c", "p90_c"},
        )
        e = _read_csv(root, "Fig1e", {"component", "share"})
        f = _read_csv(
            root,
            "Fig1f",
            {"policy", "position", "proportion", "ci_low", "ci_high", "records"},
        )
        g = _read_csv(
            root,
            "Fig1g",
            {"policy", "q10_delta_c", "q25_delta_c", "median_delta_c", "q75_delta_c", "q90_delta_c", "records"},
        )
        h = _read_csv(
            root,
            "Fig1h",
            {"reference", "weighting", "component", "loo_min_share", "loo_median_share", "loo_max_share", "n_leave_one_source_runs"},
        )

        figure, axes = create_panel_figure()
        figure.subplots_adjust(wspace=0.24, hspace=0.34)
        ax = axes.ravel()

        # a: separate record and analysis-unit flows prevent a people/record conflation.
        complete = a.set_index("population").loc["complete"]
        executable = a.set_index("population").loc["executable"]
        ax[0].set_xlim(-0.12, 1.04)
        ax[0].set_ylim(-0.4, 1.7)
        ax[0].set_xticks([])
        ax[0].set_yticks([])
        ax[0].axis("off")
        for y, key, label, loss in [
            (1.05, "records", "Records", 19),
            (0.25, "analysis_units", "Analysis units", 17),
        ]:
            values = [int(complete[key]), int(executable[key])]
            for x, value, stage in zip([0.02, 0.86], values, ["Complete", "Executable"]):
                width = 0.34 if stage == "Complete" else 0.32
                box = FancyBboxPatch(
                    (x - width / 2, y - 0.19),
                    width,
                    0.38,
                    boxstyle="round,pad=0.02",
                    facecolor="#EAF2F8" if x == 0 else "#D7EFE7",
                    edgecolor="#666666",
                    linewidth=0.65,
                )
                ax[0].add_patch(box)
                ax[0].text(x, y + 0.04, f"{value:,}", ha="center", va="center")
                ax[0].text(x, y - 0.10, stage, ha="center", va="center")
            ax[0].annotate("", xy=(0.68, y), xytext=(0.21, y), arrowprops={"arrowstyle": "->", "lw": 0.9})
            ax[0].text(0.45, y + 0.15, f"-{loss}", ha="center", va="center")
            ax[0].text(0.02, y + 0.30, label, ha="left", va="center")
        ax[0].text(0.46, 1.52, "187 buildings · 40 cities · 15 countries", ha="center", va="center")
        _finish_axis(ax[0], "a", "Archive and population", None)

        supported_centres = pd.concat([b, c], ignore_index=True)["demand_center_c"].dropna()
        vmin, vmax = float(supported_centres.min()), float(supported_centres.max())
        anchor_names = {"honolulu", "midland", "bangalore", "sydney"}
        anchor_offsets = {
            "honolulu": (4, -10),
            "midland": (3, 3),
            "bangalore": (3, -7),
            "sydney": (-4, 7),
        }
        anchor_labels = {"honolulu": "Honolulu", "midland": "UK", "bangalore": "Bengaluru", "sydney": "Sydney"}
        supported_frames = pd.concat([b, c], ignore_index=True)
        supported_frames = supported_frames.loc[
            supported_frames["supported"].astype(str).str.lower().eq("true")
        ]
        marker_unit_max = float(supported_frames["analysis_units"].max())
        for panel_axis, frame, letter, title in [
            (ax[1], b, "b", "Warm-season observations"),
            (ax[2], c, "c", "Cold-season observations"),
        ]:
            draw_world_land(panel_axis)
            is_supported = frame["supported"].astype(str).str.lower().eq("true")
            unsupported = frame.loc[~is_supported]
            panel_axis.scatter(
                unsupported["longitude"],
                unsupported["latitude"],
                s=22,
                facecolors="none",
                edgecolors=PALETTE["grey"],
                linewidths=0.8,
                zorder=2,
            )
            shown = frame.loc[is_supported].copy()
            sizes = 18 + 110 * np.sqrt(shown["analysis_units"].astype(float) / marker_unit_max)
            panel_axis.scatter(
                shown["longitude"],
                shown["latitude"],
                c=shown["demand_center_c"],
                cmap=DEMAND_CMAP,
                vmin=vmin,
                vmax=vmax,
                s=sizes,
                edgecolors="white",
                linewidths=0.45,
                zorder=3,
            )
            for row in shown.loc[shown["city"].isin(anchor_names)].itertuples():
                city = str(row.city)
                annotation = panel_axis.annotate(
                    anchor_labels[str(row.city)],
                    (row.longitude, row.latitude),
                    xytext=anchor_offsets[city],
                    textcoords="offset points",
                    ha="right" if city == "sydney" else "left",
                    va="bottom",
                    annotation_clip=True,
                )
                annotation.set_clip_on(True)
            panel_axis.text(
                0.99,
                0.02 if letter == "b" else 0.08,
                f"{len(shown)}/40 supported",
                ha="right",
                va="bottom",
                transform=panel_axis.transAxes,
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 0.6},
            )
            if letter == "b":
                colour_map = plt.get_cmap(DEMAND_CMAP)
                for colour_index in range(8):
                    panel_axis.add_patch(
                        Rectangle(
                            (-34 + 12 * colour_index, -57),
                            12,
                            5,
                            facecolor=colour_map(colour_index / 7),
                            edgecolor="none",
                            zorder=4,
                        )
                    )
                panel_axis.text(14, -48, f"{vmin:.1f}–{vmax:.1f} °C", ha="center", va="bottom")
            else:
                panel_axis.scatter([-170, -135], [-54, -54], s=[28, 82], color=PALETTE["grey"], edgecolors="white", linewidths=0.4, zorder=4)
                panel_axis.scatter([-70], [-54], s=35, facecolors="none", edgecolors=PALETTE["grey"], linewidths=0.8, zorder=4)
                panel_axis.text(-120, -54, "area=n", ha="left", va="center")
                panel_axis.text(-59, -54, "no support", ha="left", va="center")
            panel_axis.set_xticks([-120, 0, 120])
            panel_axis.set_yticks([-30, 30, 60])
            if letter == "b":
                panel_axis.set_ylabel("")
                panel_axis.tick_params(axis="y", direction="in", pad=-24)
                for tick_label in panel_axis.get_yticklabels():
                    tick_label.set_horizontalalignment("left")
            _finish_axis(panel_axis, letter, title, None)

        # d: four compact ranked bands, preserving empirical P10--P90 intervals.
        group_order = [("city", "warm"), ("city", "cold"), ("source", "warm"), ("source", "cold")]
        y_base = 0
        yticks, ylabels = [], []
        for group_index, (level, season) in enumerate(group_order):
            group = d.loc[(d["aggregation_level"] == level) & (d["season"] == season)].sort_values("p50_c")
            count = len(group)
            ys = np.linspace(y_base, y_base + 0.68, count) if count > 1 else np.array([y_base + 0.34])
            ax[3].hlines(ys, group["p10_c"], group["p90_c"], color=PALETTE["sky"], linewidth=0.7)
            ax[3].scatter(group["p50_c"], ys, s=5, color=PALETTE["blue"], zorder=3)
            widths = group["p90_c"] - group["p10_c"]
            yticks.append(y_base + 0.34)
            ylabels.append(f"{'C' if level == 'city' else 'S'}/{season[0]}: {count}, {widths.median():.1f}°")
            y_base += 1.0
        ax[3].set_yticks([])
        ax[3].set_xlim(14, 32)
        ax[3].set_xticks([16, 20, 24, 28, 32])
        for yy, label in zip(yticks, ylabels):
            ax[3].text(14.15, yy, label, ha="left", va="center")
        ax[3].set_xlabel("Demand temperature (°C)", labelpad=-4.5)
        _finish_axis(ax[3], "d", "Within-place demand ranges", "x")

        # e: exact mutually exclusive variance allocation.
        component_labels = {
            "country": "Country",
            "city": "City",
            "release_building_group_id": "Building",
            "release_unit_id": "Within building",
            "within_unit": "Repeat",
        }
        order = ["country", "city", "release_building_group_id", "release_unit_id", "within_unit"]
        colors = VARIANCE_COLORS
        lookup = e.set_index("component")["share"]
        left = 0.0
        for component, color in zip(order, colors):
            value = float(lookup.loc[component]) * 100
            ax[4].barh([0], [value], left=left, color=color, height=0.45)
            offset_y = {"city": 0.13, "release_building_group_id": -0.13}.get(component, 0)
            ax[4].text(left + value / 2, offset_y, f"{value:.1f}%", ha="center", va="center")
            left += value
        ax[4].set_xlim(0, 100)
        ax[4].set_ylim(-0.65, 0.85)
        ax[4].set_yticks([])
        ax[4].set_xlabel("Share of variance (%)", labelpad=-4.5)
        handles = [Patch(facecolor=color, label=component_labels[name]) for name, color in zip(order, colors)]
        ax[4].legend(handles=handles, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0), handlelength=0.8, columnspacing=0.7, handletextpad=0.3)
        _finish_axis(ax[4], "e", "Where demand variance resides", "x")

        # f: each policy keeps its own denominator and within-range CI.
        positions = ["below", "within", "above"]
        position_colors = {"below": PALETTE["blue"], "within": PALETTE["light_grey"], "above": PALETTE["vermillion"]}
        policy_order = list(dict.fromkeys(f["policy"]))
        compact_policy = {
            "Australia model COP": "AUS COP",
            "Brazil NR-17": "BRA NR17",
            "Canada CCOHS": "CAN CCOHS",
            "India ECSBC 2024": "IND ECSBC",
            "Indonesia Permenkes 48": "IDN P48",
            "Malaysia ICOP IAQ": "MYS ICOP",
            "Singapore SS 553": "SGP SS553",
            "Sweden AFS 2023:12": "SWE AFS",
            "USA OSHA guidance": "USA OSHA",
        }
        for y, policy in enumerate(policy_order):
            group = f.loc[f["policy"] == policy].set_index("position")
            left = 0.0
            for position in positions:
                value = float(group.loc[position, "proportion"])
                ax[5].barh(y, value, left=left, color=position_colors[position], height=0.68)
                left += value
            within = group.loc["within"]
            proportion = float(within["proportion"])
            lower = proportion - float(within["ci_low"])
            upper = float(within["ci_high"]) - proportion
            ax[5].errorbar(proportion, y, xerr=np.array([[lower], [upper]]), color="#222222", capsize=1.5, linewidth=0.8)
            ax[5].text(1.04, y, f"{float(within['proportion'])*100:.0f}%", ha="left", va="center")
            ax[5].text(1.50, y, f"{int(within['records']):,}", ha="right", va="center")
        ax[5].set_yticks([])
        for y, policy in enumerate(policy_order):
            ax[5].text(-0.02, y, compact_policy[policy], ha="right", va="center")
        ax[5].set_ylim(len(policy_order) - 0.5, -2.25)
        ax[5].set_xlim(-0.46, 1.55)
        ax[5].set_xticks([0, 0.5, 1.0], ["0", "50", "100"])
        ax[5].set_xlabel("Demand records by range (%)", labelpad=-4.5)
        handles = [Patch(facecolor=position_colors[p], label=p.title()) for p in positions]
        ax[5].legend(handles=handles, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.99), handlelength=0.8, columnspacing=0.7, handletextpad=0.3)
        _finish_axis(ax[5], "f", "Policy-range coverage", "x")

        # g: empirical demand-minus-target quantiles.
        y = np.arange(len(g))
        ax[6].axvline(0, color="#333333", linewidth=0.7)
        for index, row in enumerate(g.itertuples()):
            ax[6].hlines(index, row.q10_delta_c, row.q90_delta_c, color=PALETTE["sky"], linewidth=1.2)
            ax[6].hlines(index, row.q25_delta_c, row.q75_delta_c, color=PALETTE["blue"], linewidth=4)
            ax[6].scatter(row.median_delta_c, index, s=26, color=PALETTE["vermillion"], zorder=3)
            label_y = index + 0.14 if index < len(g) - 1 else index - 0.14
            value_label = ax[6].text(
                2.55,
                label_y,
                f"{row.median_delta_c:+.1f}° (n={int(row.records)})",
                ha="right",
                va="top" if index < len(g) - 1 else "bottom",
                clip_on=True,
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.78, "pad": 0.15},
            )
            value_label.set_gid("fig1g-value-label")
        ax[6].set_yticks(y, [])
        for index, value in enumerate(g["policy"]):
            label = value.replace("China summer rule", "China").replace("Japan Cool Biz", "Japan").replace("Thailand EGAT campaign", "Thailand")
            ax[6].text(-5.05, index, label, ha="left", va="center")
        ax[6].invert_yaxis()
        ax[6].set_xlim(-5.2, 4.8)
        ax[6].set_xticks([-4, -2, 0, 2, 4])
        ax[6].set_xlabel("Demand centre − campaign target (°C)")
        _finish_axis(ax[6], "g", "Guidance relative to demand", "x")

        # h: all 12 coefficient/weighting specifications remain visible.
        component_names = {
            "between_analysis_unit_within_building": "Within bldg.",
            "between_country": "Country",
            "between_city_within_country": "City",
            "between_building_within_city": "Building",
            "within_analysis_unit": "Repeat",
        }
        comp_order = list(component_names)
        for y, component in enumerate(comp_order):
            group = h.loc[h["component"] == component]
            for _, row in group.iterrows():
                ax[7].hlines(y, row["loo_min_share"] * 100, row["loo_max_share"] * 100, color="#C9C9C9", linewidth=0.6)
                ax[7].scatter(row["loo_median_share"] * 100, y, s=10, color=PALETTE["blue"], alpha=0.8)
            ax[7].hlines(y, group["loo_min_share"].min() * 100, group["loo_max_share"].max() * 100, color=PALETTE["vermillion"], linewidth=2.2)
            ax[7].scatter(group["loo_median_share"].median() * 100, y, s=28, color=PALETTE["vermillion"], zorder=3)
        ax[7].set_yticks([])
        for y, component in enumerate(comp_order):
            ax[7].text(-1.5, y, component_names[component], ha="right", va="center")
        ax[7].invert_yaxis()
        ax[7].set_xlim(-20, 75)
        ax[7].set_xticks([0, 25, 50, 75])
        ax[7].set_xlabel("Variance share (%)")
        _finish_axis(ax[7], "h", "Robustness: 12 specs × 27 removals", "x")

        return figure

    CAPTIONS = {
        1: (
            "**Figure 1 | Observed human demand is geographically structured but dominated by "
            "within-building heterogeneity.** **a,** The complete office archive contains 21,418 records "
            "from 12,130 analysis units; the executable set retains 21,399 records from 12,113 "
            "analysis units in 187 buildings, 40 cities and 15 countries. **b,c,** Observed-city warm- "
            "and cold-season demand centres. Filled circles denote season-supported cities, hollow "
            "circles observed but unsupported cities, circle area the number of analysis units and "
            "colour the demand centre on the same 19.5–30.6 °C scale; unsampled geography is left "
            "blank and no spatial interpolation is applied. **d,** Empirical P10–P90 intervals and "
            "medians for city- and source-level warm/cold aggregates under analysis-unit-equal "
            "weighting; labels give the number of aggregates and median interval width. **e,** "
            "Hierarchical variance allocation across country (22.3%), city within country (7.7%), "
            "building within city (5.8%), analysis unit within building (56.1%) and repeated "
            "measurements (8.2%). **f,** Weighted demand records below, within and above nine bilateral "
            "policy or design ranges at Griffiths coefficient 0.50; labels give the within-range "
            "share and eligible-record denominator, and whiskers are nested-bootstrap 95% confidence "
            "intervals for the within-range share. **g,** Empirical P10–P90 (thin), P25–P75 (thick) and "
            "median (point) of demand centre minus campaign target for three prespecified campaigns; "
            "positive values indicate a demand centre warmer than the target, and negative values "
            "indicate a demand centre cooler than the target. **h,** Leave-one-source sensitivity ranges "
            "and medians across 12 coefficient/weighting specifications, each based on 27 source "
            "removals. Policy documents in **f,g** are external temperature coordinates rather than "
            "individual comfort ground truth."
        ),
        2: (
            "**Figure 2 | Independent shared-room votes reveal demand disagreement hidden by "
            "averages.** **a,** Prespecified construction of the NUS 30-min validation cohort from "
            "4,378 raw rows to 63 temperature-homogeneous room-windows containing 225 valid "
            "person-window votes. **b,** Cooler, no-change and warmer vote shares in all 63 "
            "room-windows, ordered by net vote and disagreement; 49 windows contain more than one "
            "response category and six contain warmer and cooler requests within the same 30-min "
            "room-window. **c,** Vote counts in the six room-windows with within-window opposing "
            "requests; grey circles denote no-change votes and labels give vote count, mean room "
            "temperature and within-window temperature span. **d,** Full request burden, mean "
            "absolute vote, versus burden after averaging, absolute mean vote, for each window. "
            "Hollow orange points are exact cancellations; the dashed line is equality and the "
            "median compression gap is zero despite a positive mean gap. **e,** The four exact-"
            "cancellation windows, in which non-zero full burden is represented as zero after "
            "averaging. **f,** False acceptance and false rejection at burden thresholds 0, 0.25, "
            "0.50 and 0.75; all four thresholds retain zero false rejections and 2–5 false "
            "acceptances among 63 windows. **g,** Disagreement, within-window opposing requests and "
            "mean Jensen gap under 15-, 30- and 60-min windows; whiskers are crossed room/date "
            "cluster-bootstrap 95% confidence intervals and n is the eligible-window denominator. "
            "**h,** Independent ENTH replication (34 windows, 81 votes) beside NUS (63 windows, 225 "
            "votes). Windows, rather than votes, are the inferential unit throughout."
        ),
        3: (
            "**Figure 3 | Demand location redistributes annual energy and carbon under three "
            "controls.** **a,** Uniform average, personal centre and personal range control "
            "families generate physical actions in the same building under identical operating "
            "permissions. These controls are distinct from the mean and full-distribution "
            "qualification representations used to score the same actions. **b,** Assigned P10–P90 "
            "demand intervals, centres and seed variation for three population compositions in four "
            "building/climate contexts. **c,** Prespecified orthogonal transformation "
            "T′=μ+Δ+λ(T−μ), with demand-location shift Δ from −2 to +2 °C and spread scale λ from 0 "
            "to 1.5; the open point is the Δ=0, λ=1 reference. **d,e,** Median and interquartile "
            "range of matched service-qualified changes in purchased space-HVAC site energy and "
            "operational CO₂ relative to the reference. The coloured denominator tables give "
            "matched n for Miami, New Delhi, Port Angeles and Seattle in left-to-right Δ order (−2, "
            "−1, +1 and +2 °C). **f,** Median signed target-minus-reference changes in heating "
            "electricity, cooling electricity, fan electricity and gas HHV for Δ=+2 °C, using the "
            "amended reference/target carrier fields directly; diamonds show median total-energy "
            "change and the two-line context labels give 22, 20, 8 and 9 matched cells. The artwork "
            "states the full accounting boundary of 59 matched and 85 unmatched design rows; "
            "unmatched rows contribute no fabricated delta. **g,** Energy direction for all 130 "
            "default uniform-versus-alternative paired definitions, separated by control and "
            "context: 54 increases, 3 decreases, 36 ties, 7 direction-unstable comparisons and 30 "
            "unmatched comparisons. **h,** Outcome shares across Griffiths energy, Griffiths carbon "
            "and personal range-width sensitivity families with explicit denominators and the same outcome "
            "colour/hatch key as panel g. Population assignments are controlled archive scenarios "
            "rather than claims of local population representativeness; carbon factors are fixed "
            "regional scenarios."
        ),
    }

    ALT_TEXT = {
        1: (
            "Eight-panel portrait figure. An archive flow retains 21,399 of 21,418 records. Two "
            "observed-city maps show 31 warm-season and 26 cold-season supported cities without "
            "interpolation. Ranked empirical intervals show broad within-city and within-source "
            "demand ranges. A variance bar assigns the largest share, 56.1%, to analysis units "
            "within buildings. Policy-range bars show wide variation in observed-demand coverage. "
            "Panel g plots demand centre minus campaign target: positive values mean that the demand "
            "centre is warmer than the target, and negative values mean that it is cooler. "
            "Leave-one-source sensitivity preserves the importance of within-building heterogeneity, "
            "while showing that its magnitude depends on weighting and the Griffiths coefficient."
        ),
        2: (
            "Eight-panel portrait figure. A filtering flow yields 63 shared-room windows and 225 "
            "direct votes. A 63-row composition chart shows disagreement in 49 windows, including "
            "six with warmer and cooler requests within the same 30-min room-window. Six diverging "
            "bars detail those within-window opposing requests. A scatter shows that averaging can "
            "only preserve or reduce represented request burden and exactly cancels four non-neutral "
            "windows. Threshold bars show false acceptance but no false rejection. Results persist "
            "across 15-, 30- and 60-min windows and replicate in a second ENTH dataset, although "
            "confidence intervals are wider there."
        ),
        3: (
            "Eight-panel portrait figure. Panel a compares three control families—Uniform average, "
            "personal centre and personal range controls—under the same building and operating "
            "permissions. Mean and full-distribution are separate qualification representations "
            "used to score generated actions, not additional controls. Population scenarios differ "
            "in both demand centre and width. A centre-by-spread grid defines independent "
            "perturbations. Energy and carbon response curves change sign across contexts: warmer "
            "demand reduces median annual energy and carbon in Miami and New Delhi but increases "
            "them in Port Angeles and Seattle. Coloured tables above both curves report the matched "
            "denominator at each demand shift without covering the data. Carrier bars show "
            "concurrent shifts among heating electricity, cooling electricity, fans and gas and "
            "state that 59 rows match while 85 remain unmatched. Stacked outcome bars retain "
            "increases, decreases, ties, unstable directions and unmatched cases. Sensitivity bars "
            "reuse that outcome key and show that direction and match availability depend on demand "
            "assumptions, so personalization does not imply universal savings."
        ),
    }
    from render_revised_main_figures import restructure, readable_display_labels
    from thermal_heterogeneity.figures import apply_figure_style, apply_data_axis_style
    DPI = 600
    apply_figure_style()
    PANEL_ROOT = RESULTS / "panel_data"
    figure = build_figure_1(PANEL_ROOT)
    figure._stage7_panel_root = PANEL_ROOT
    figure._stage7_decision_value = RESULTS / "decision_value/MAIN_FIG6G_Q5_REGRET.csv"
    restructure(figure, 1)
    readable_display_labels(figure, 1)
    # Compact text keeps the data frames wide while preserving equal columns.
    result_axes = sorted(
        (axis for axis in figure.axes if axis.get_title(loc="left").startswith(tuple(f"({letter})" for letter in "abcdefgh"))),
        key=lambda axis: axis.get_title(loc="left")[1],
    )
    for index, axis in enumerate(result_axes):
        position = axis.get_position()
        y0 = position.y0 - .015 if index < 2 else position.y0
        axis.set_position((.16 if index % 2 == 0 else .64, y0, .34, position.height))
    variance_axis = result_axes[3]
    variance_axis.set_yticks(variance_axis.get_yticks(), [
        label.get_text().replace("Within building", "Within\nbuilding")
        for label in variance_axis.get_yticklabels()
    ])
    result_axes[7].set_yticks([0, 1], ["Singapore", "Longitudinal"])
    for text in list(figure.texts):
        if "supported" in text.get_text() or text.get_text() == "°C":
            text.remove()
    for axis in result_axes[:2]:
        for artist in list(axis.texts):
            if artist.get_text() in {"area=n", "marker area scales with n", "no support"}:
                artist.remove()
    for axis, count in zip(result_axes[:2], [31, 26]):
        axis.text(.975, .955, f"{count}/40 cities", transform=axis.transAxes,
                  ha="right", va="top", fontsize=11, clip_on=True,
                  bbox={"facecolor": "white", "edgecolor": "none", "alpha": .88, "pad": .4})
    result_axes[1].set_ylabel("")
    result_axes[1].tick_params(axis="y", left=False, labelleft=False)
    colorbar_axis = next(axis for axis in figure.axes if getattr(axis, "_colorbar", None) is not None)
    old_colorbar = colorbar_axis._colorbar
    colorbar_norm, colorbar_cmap = old_colorbar.norm, old_colorbar.cmap
    colorbar_ticks = list(old_colorbar.get_ticks())
    map_position = result_axes[0].get_position()
    colorbar_axis.remove()
    colorbar_axis = figure.add_axes([.55, map_position.y0 + .012, .013,
                                     max(.04, map_position.height - .024)])
    new_colorbar = figure.colorbar(
        plt.cm.ScalarMappable(norm=colorbar_norm, cmap=colorbar_cmap),
        cax=colorbar_axis, orientation="vertical",
    )
    new_colorbar.set_ticks(colorbar_ticks)
    new_colorbar.ax.set_ylabel("Demand centre (°C)", fontsize=11, rotation=270,
                               va="bottom", labelpad=8)
    new_colorbar.ax.tick_params(labelsize=11, pad=2)
    # The six simulated demand compositions retain the actual 267 assigned profiles.
    # This replaces the repeated variance sensitivity; its full evidence remains in SI.
    composition_path = RESULTS / "Joint_main/DEMAND_COMPOSITION_PROFILES.csv"
    composition_profiles = pd.read_csv(composition_path)
    required = {"composition_id", "profile_id", "centre_c", "lower_c", "upper_c"}
    if (not required.issubset(composition_profiles.columns) or len(composition_profiles) != 1602
            or composition_profiles.composition_id.nunique() != 6
            or composition_profiles.duplicated(["composition_id", "profile_id"]).any()
            or not composition_profiles.groupby("composition_id").size().eq(267).all()):
        raise ValueError("Figure 1g requires six complete fixed demand compositions with 267 profiles each")
    composition_profiles["width_c"] = composition_profiles.upper_c - composition_profiles.lower_c
    if (composition_profiles.width_c <= 0).any():
        raise ValueError("Every displayed Personal range must have positive width")
    demand_panel = next(axis for axis in figure.axes if axis.get_title(loc="left").startswith("(g)"))
    demand_panel.clear()
    demand_panel.set_axis_off()
    demand_panel.set_xticks([])
    demand_panel.set_yticks([])
    demand_panel.set_title("(g) Simulated demand", loc="left", fontsize=12, weight="bold", pad=9)
    centre_axis = demand_panel.inset_axes([0, 0, .43, 1])
    width_axis = demand_panel.inset_axes([.61, 0, .39, 1])
    source_rows = []
    composition_ids = sorted(composition_profiles.composition_id.unique())
    for index, composition_id in enumerate(composition_ids):
        group = composition_profiles[composition_profiles.composition_id.eq(composition_id)]
        for child, metric, colour in [(centre_axis, "centre_c", "#6F9EBC"),
                                      (width_axis, "width_c", "#C78470")]:
            q10, q25, q50, q75, q90 = np.quantile(group[metric], [.1, .25, .5, .75, .9])
            child.hlines(index, q10, q90, color=colour, linewidth=1.1)
            child.hlines(index, q25, q75, color=colour, linewidth=3.3)
            child.scatter(q50, index, s=15, color=colour, edgecolors="white", linewidths=.35, zorder=4)
            source_rows.append({"composition_id": int(composition_id), "composition_label": (f"ASHRAE {index + 1}" if index < 3 else f"CTCD {index - 2}"),
                                "profiles": len(group), "metric": metric, "p10": q10, "p25": q25,
                                "p50": q50, "p75": q75, "p90": q90})
    for child in [centre_axis, width_axis]:
        child.set_ylim(5.65, -.65)
        child.tick_params(axis="both", labelsize=11, pad=3, length=2.5)
        child.spines["right"].set_visible(False)
        child.spines["top"].set_visible(False)
        child.spines["left"].set_visible(False)
        child.tick_params(axis="y", length=0)
        child.set_yticks(range(6))
    centre_axis.set_yticklabels(["ASHRAE 1", "ASHRAE 2", "ASHRAE 3", "CTCD 1", "CTCD 2", "CTCD 3"])
    width_axis.set_yticklabels([])
    centre_axis.set_xlim(15, 34)
    centre_axis.set_xticks([16, 24, 32])
    width_axis.set_xlim(0, 13)
    width_axis.set_xticks([0, 6, 12])
    centre_axis.set_xlabel("Centre (°C)", fontsize=11, labelpad=6)
    width_axis.set_xlabel("Width (°C)", fontsize=11, labelpad=6)
    composition_summary = INTERMEDIATE / LABEL
    composition_summary.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(source_rows).to_csv(composition_summary / "Figure_01g_composition_quantiles.csv", index=False)
    (RESULTS/"captions"/(LABEL+".txt")).write_text('Figure 1 | Thermal demand across places and shared rooms. Panels a–f use ASHRAE office observations; g uses both harmonized archives. a,b, Reconstructed city demand centres in warm/cold seasons. Marker area gives analysis-unit count on one shared scale and colour gives temperature; hollow markers indicate unsupported summaries. Map labels count supported city summaries: 31/40 warm and 26/40 cold. c, City/source medians and 10th–90th percentiles. d, Unit-weighted variance across 187 building groups (150 original, 37 inferred identifiers). e, Record-equal shares below, within and above nine policy/design ranges, with nested-bootstrap 95% intervals for within-range coverage. f, Demand centre minus campaign target. g, Six simulation inputs of 267 profiles each: reference centres (blue) and reference demand-interval widths (terracotta), on separate scales. Thin/thick segments in f,g show 10th–90th/25th–75th percentiles; points show medians. h, Direct requests from the Singapore (63 windows) and longitudinal (34 windows) studies. Categories are identical (Same), different without opposing requests (Mixed), and simultaneous warmer/cooler (Opposed). Different requests equal Mixed plus Opposed: 43 + 6 = 49 and 20 + 2 = 22, respectively.', encoding="utf-8")
    apply_data_axis_style(figure, map_axes=figure.axes[:2] if 1 == 1 else [])
    save_plot(figure, LABEL, dpi=DPI, exact_pixels=True)


draw_figure_01()


In [ ]:

# Figure_02: three controls, service delivered and information needed.
def draw_figure_02():
    annual = pd.read_csv(RESULTS / "final_submission/evidence/FINAL_ANNUAL_COMPARISONS.csv")
    direct = annual[(annual.quality_scope == "primary") & (annual.comparison_mode == "direct") &
                    (annual.permission == "native_zone") & annual.layout.isin(["random_mixing", "demand_ordered"]) &
                    (annual.status == "matched")].copy()
    info = json.loads((RESULTS / "decision_value/INFORMATION_HIERARCHY_SUMMARY.json").read_text())["zone_mean_to_full_profile"]["counts"]
    fig = plt.figure(figsize=(180/25.4, 175/25.4))
    gs = fig.add_gridspec(3, 2, height_ratios=[0.78, 1, 1], hspace=0.58, wspace=0.38)
    ax = fig.add_subplot(gs[0, :]); ax.axis("off"); final_title(ax, "a", "Three controls use different demand information")
    cards = [
        (0.03, "Uniform average", "One building-wide\ndemand centre", CONTROL_COLORS["uniform_average"]),
        (0.355, "Personal centre", "Assigned centres guide\neach command group", CONTROL_COLORS["personal_center"]),
        (0.68, "Personal range", "Assigned ranges guide\neach command group", CONTROL_COLORS["personal_range"]),
    ]
    for x, title, text, color in cards:
        ax.add_patch(FancyBboxPatch((x, .18), .285, .52, boxstyle="round,pad=0.012,rounding_size=0.018",
                                    fc=color+"26", ec=color, lw=1.2, transform=ax.transAxes))
        ax.text(x+.142, .55, title, ha="center", va="center", fontweight="bold", transform=ax.transAxes)
        ax.text(x+.142, .34, text, ha="center", va="center", fontsize=9, transform=ax.transAxes)
    ax.annotate("Same building, weather, equipment and physical occupant loads", xy=(.5,.05), ha="center", fontsize=9, xycoords="axes fraction")

    metrics = [
        ("delta_mean_reference_mismatch_c", "b", "Population mismatch", "Mismatch reduction (°C)", -1),
        ("delta_fixed_tail_reference_mismatch_c", "c", "Least-served group mismatch", "Mismatch reduction (°C)", -1),
        ("delta_modeled_interval_person_hour_coverage", "d", "Personal range coverage", "Coverage gain\n(percentage points)", 100),
    ]
    axes = [fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1]), fig.add_subplot(gs[2,0])]
    rng = np.random.default_rng(2401)
    for ax, (column, letter, title, ylabel, scale) in zip(axes, metrics):
        values = direct[column] * scale
        for j, control in enumerate(("personal_center","personal_range")):
            part = direct[direct.alternative == control]
            y = part[column] * scale
            x = np.full(len(part), j) + rng.normal(0, .045, len(part))
            for source, marker in SOURCE_MARKERS.items():
                mask = part.data_source == source
                ax.scatter(x[mask], y[mask], s=24, marker=marker, facecolor=CONTROL_COLORS[control],
                           edgecolor="white", lw=.45, alpha=.82)
            ax.plot([j-.13,j+.13], [np.median(y),np.median(y)], color="black", lw=1.5)
        ax.axhline(0,color="#9D9D9D",lw=.8)
        ax.set_xticks([0,1], ["Personal\ncentre","Personal\nrange"])
        ax.set_ylabel(ylabel, labelpad=3); final_title(ax, letter, title); final_style(ax)
    fig.legend([Line2D([0],[0],marker="o",ls="",color="#777",label="ASHRAE"),
                Line2D([0],[0],marker="s",ls="",color="#777",label="CTCD")],
               ["ASHRAE","CTCD"],frameon=False,ncol=2,loc="upper center",bbox_to_anchor=(.5,.72))

    ax = fig.add_subplot(gs[2,1]); final_title(ax,"e","Averaging changes plan qualification")
    safe, required, unresolved = info["zone_mean_safe_cells"], info["distribution_required_cells"], info["finite_library_unresolved_cells"]
    ax.barh([2],[safe],color="#9FC1CF",height=.52)
    ax.barh([2],[required],left=[safe],color="#C98B77",height=.52)
    ax.barh([1],[info["zone_mean_selected_plan_fails_full_review_cells"]],left=[safe],color="#B76855",height=.52)
    ax.barh([0],[safe+required],color="#AAB9BC",height=.52)
    ax.barh([0],[unresolved],left=[safe+required],color="#D8D8D8",height=.52,hatch="//",edgecolor="white")
    ax.text(safe/2,2,f"Mean\nalone\n{safe}",ha="center",va="center",fontsize=8.5,linespacing=.9)
    ax.text(safe+required/2,2,f"Full demand\nneeded\n{required}",ha="center",va="center",fontsize=8.5,linespacing=.9)
    ax.text(safe+info["zone_mean_selected_plan_fails_full_review_cells"]/2,1,"250",ha="center",va="center")
    ax.text((safe+required)/2,0,"606 resolvable",ha="center",va="center")
    ax.text(safe+required+unresolved/2,0,"402 unresolved",ha="center",va="center")
    ax.set_yticks([2,1,0],["Resolvable","Zone-mean\nselection failed","All tested"])
    ax.set_xlabel("Decision settings (count)"); final_style(ax)
    save_plot(fig,"Figure_02",size_mm=(180,175))
draw_figure_02()


In [ ]:

# Figure_03: service, energy, carbon and seasonal reallocation.
def draw_figure_03():
    annual = pd.read_csv(RESULTS / "final_submission/evidence/FINAL_ANNUAL_COMPARISONS.csv")
    selections = pd.read_csv(RESULTS / "final_submission/evidence/FINAL_ANNUAL_SELECTIONS.csv")
    seasonal = pd.read_csv(RESULTS / "final_submission/evidence/FINAL_SEASONAL_DIFFERENCES.csv")
    direct = annual[(annual.quality_scope=="primary")&(annual.comparison_mode=="direct")&(annual.permission=="native_zone")&annual.layout.isin(["random_mixing","demand_ordered"])&(annual.status=="matched")].copy()
    common = annual[(annual.quality_scope=="primary")&(annual.comparison_mode=="equal_service")&(annual.service_fraction==1)&annual.layout.isin(["random_mixing","demand_ordered"])&(annual.status=="matched")].copy()
    fig,axs=plt.subplots(3,2,figsize=(180/25.4,190/25.4)); plt.subplots_adjust(hspace=.62,wspace=.38)
    ax=axs[0,0]; final_title(ax,"a","Service and annual energy")
    for (source,control),g in direct.groupby(["data_source","alternative"]):
        ax.scatter(-g.delta_mean_reference_mismatch_c, g.delta_site_pct, s=25, marker=SOURCE_MARKERS[source],color=CONTROL_COLORS[control],edgecolor="white",lw=.4,alpha=.85)
    ax.axvline(0,color="#A0A0A0",lw=.8);ax.axhline(0,color="#A0A0A0",lw=.8);ax.set_xlabel("Reduction in population mismatch (°C)");ax.set_ylabel("Annual energy change (%)");final_style(ax)
    ax.text(.98,.96,"34 increased\n35 decreased",transform=ax.transAxes,ha="right",va="top",fontweight="bold")
    ax=axs[0,1]; final_title(ax,"b","Energy and operational carbon")
    for (source,control),g in direct.groupby(["data_source","alternative"]):
        ax.scatter(g.delta_annual_site_kwh/1000,g.delta_annual_operational_co2_kg/1000,s=25,marker=SOURCE_MARKERS[source],color=CONTROL_COLORS[control],edgecolor="white",lw=.4)
    ax.axvline(0,color="#A0A0A0",lw=.8);ax.axhline(0,color="#A0A0A0",lw=.8);ax.set_xlabel("Annual energy change (MWh)");ax.set_ylabel("Operational carbon change (tCO₂)");final_style(ax)
    ax=axs[1,0]; final_title(ax,"c","Where energy is added or avoided")
    components=[("Heating electricity","delta_annual_heating_electric_kwh","#C9806A"),("Cooling electricity","delta_annual_cooling_electric_kwh","#6E9CB7"),("Fans","delta_annual_fans_electric_kwh","#9A849E"),("Heating gas","delta_annual_gas_hhv_kwh","#C9A464")]
    for i,(label,col,color) in enumerate(components):
        vals=direct[col]/1000; q1,med,q3=vals.quantile([.25,.5,.75]); ax.vlines(i,q1,q3,color=color,lw=8,alpha=.55);ax.scatter(i,med,s=42,color=color,edgecolor="white",zorder=3)
    ax.set_xticks(range(4),[x[0].replace(" ","\n") for x in components]);ax.set_ylabel("Median change\n(MWh/year)", labelpad=2);ax.axhline(0,color="#999",lw=.8);final_style(ax)
    ax=axs[1,1]; final_title(ax,"d","Energy under common annual limits")
    rng=np.random.default_rng(33)
    for j,control in enumerate(("personal_center","personal_range")):
        p=common[common.alternative==control];x=np.full(len(p),j)+rng.normal(0,.045,len(p));
        for source,marker in SOURCE_MARKERS.items():
            m=p.data_source==source;ax.scatter(x[m],p.loc[m,"delta_site_pct"],s=20,marker=marker,color=CONTROL_COLORS[control],alpha=.72,edgecolor="white",lw=.3)
        ax.plot([j-.14,j+.14],[p.delta_site_pct.median()]*2,color="black",lw=1.4)
    ax.set_xticks([0,1],["Personal\ncentre","Personal\nrange"]);ax.set_ylabel("Energy change (%)");ax.axhline(0,color="#999",lw=.8);final_style(ax);ax.text(.02,.96,"Median: 0.0%",transform=ax.transAxes,va="top",fontweight="bold")
    ax=axs[2,0];final_title(ax,"e","Options under tighter service limits")
    tight=selections[(selections.quality_scope=="primary")&(selections.comparison_mode=="equal_service")&(selections.service_fraction==.5)&selections.layout.isin(["random_mixing","demand_ordered"])]
    counts=tight[tight.selection_status=="selected"].groupby("control").size().reindex(CONTROL_ORDER,fill_value=0)
    ax.bar(range(3),counts.values,color=[CONTROL_COLORS[c] for c in CONTROL_ORDER],width=.65);ax.set_xticks(range(3),["Uniform\naverage","Personal\ncentre","Personal\nrange"]);ax.set_ylabel("Selected conditions / 144");final_style(ax)
    for i,v in enumerate(counts):ax.text(i,v+.25,str(v),ha="center",fontweight="bold")
    ax=axs[2,1];final_title(ax,"f","Seasonal reallocation")
    s=seasonal[(seasonal.quality_scope=="primary")&(seasonal.comparison_mode=="direct")&(seasonal.permission=="native_zone")&seasonal.layout.isin(["random_mixing","demand_ordered"])]
    order=["DJF","MAM","JJA","SON"]
    for _,g in s.groupby(["data_source","context","composition_id","layout","control"]):
        gg=g.set_index("season").reindex(order);ax.plot(range(4),gg.delta_site_kwh/1000,color="#C8C8C8",lw=.55,alpha=.45)
    for control in ("personal_center","personal_range"):
        med=s[s.control==control].groupby("season").delta_site_kwh.median().reindex(order)/1000
        ax.plot(range(4),med,color=CONTROL_COLORS[control],marker="o",lw=2,label=CONTROL_LABELS[control])
    ax.set_xticks(range(4),order);ax.set_ylabel("Energy change (MWh/season)");ax.axhline(0,color="#999",lw=.8);final_style(ax);ax.legend(frameon=False)
    fig.legend(handles=[Line2D([0],[0],marker="o",ls="",color="#777",label="ASHRAE"),Line2D([0],[0],marker="s",ls="",color="#777",label="CTCD")],loc="upper center",ncol=2,frameon=False,bbox_to_anchor=(.5,.995))
    save_plot(fig,"Figure_03",size_mm=(180,190))
draw_figure_03()


In [ ]:

# Figure_04: placement and command permission as separate levers.
def draw_figure_04():
    perm=pd.read_csv(RESULTS/'final_submission/evidence/FINAL_PERMISSION_EFFECTS.csv')
    place=pd.read_csv(RESULTS/'final_submission/evidence/FINAL_PLACEMENT_EFFECTS.csv')
    perm=perm[(perm.quality_scope=='primary')&perm.layout.isin(['random_mixing','demand_ordered'])&perm.control.isin(['personal_center','personal_range'])]
    place=place[(place.quality_scope=='primary')&place.permission.eq('native_zone')&place.control.isin(['personal_center','personal_range'])]
    fig=plt.figure(figsize=(180/25.4,188/25.4));gs=fig.add_gridspec(3,2,height_ratios=[.84,1,1],hspace=.70,wspace=.36)
    ax=fig.add_subplot(gs[0,:]);ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis('off');final_title(ax,'a','Placement and command permission are separate choices')

    cool='#7FA7BA'; mid='#C6A064'; warm='#C98B77'; person_colors=[cool]*6+[mid]*6+[warm]*6
    random_groups=[[0,6,12,1,7,13],[2,8,14,3,9,15],[4,10,16,5,11,17]]
    ordered_groups=[list(range(0,6)),list(range(6,12)),list(range(12,18))]
    def placement(origin,groups,label):
        ax.text(origin+.115,.77,label,ha='center',va='center',fontweight='bold')
        for zone,ids in enumerate(groups):
            x0=origin+zone*.078
            ax.add_patch(Rectangle((x0,.28),.068,.37,fc='#F7F7F5',ec='#8A8A8A',lw=.8))
            for pos,idx in enumerate(ids):
                x=x0+.018+(pos%2)*.031; y=.58-(pos//2)*.105
                ax.scatter(x,y,s=31,color=person_colors[idx],edgecolor='white',linewidth=.35,zorder=3)
            ax.text(x0+.034,.22,f'Zone {zone+1}',ha='center',va='center',fontsize=9)
    placement(.02,random_groups,'Random mixing')
    placement(.275,ordered_groups,'Demand ordered')
    ax.annotate('',xy=(.268,.47),xytext=(.255,.47),arrowprops=dict(arrowstyle='->',color='#777777',lw=1))

    def permission(origin,label,mode):
        ax.text(origin+.085,.77,label,ha='center',fontweight='bold')
        boxes=[]
        for r in range(2):
            for c in range(3):
                x=origin+c*.054; y=.35+r*.16
                ax.add_patch(Rectangle((x,y),.047,.12,fc='#E5EFEA',ec='#88A899',lw=.7))
                boxes.append((x,y))
        if mode=='building':
            ax.add_patch(Rectangle((origin-.008,.33),.171,.32,fill=False,ec='#6F9B86',lw=1.6))
        elif mode=='floor':
            for y in (.34,.50): ax.add_patch(Rectangle((origin-.006,y),.167,.14,fill=False,ec='#6F9B86',lw=1.35))
        else:
            for x,y in boxes: ax.add_patch(Rectangle((x-.003,y-.01),.053,.14,fill=False,ec='#6F9B86',lw=1.05))
        ax.text(origin+.078,.22,{'building':'1 command','floor':'2 commands','zone':'6 commands'}[mode],ha='center',fontsize=9)
    permission(.54,'Building','building');permission(.715,'Floor','floor');permission(.89,'Native zone','zone')
    ax.text(.17,.07,'Same people and zone boundaries',ha='center',fontsize=9)
    ax.text(.80,.07,'Same layout; only command grouping changes',ha='center',fontsize=9)

    rng=np.random.default_rng(45)
    panels=[('building_to_native_zone','delta_modeled_interval_person_hour_coverage',100,'b','Permission and service','Coverage change\n(percentage points)'),('building_to_native_zone','delta_annual_site_kwh',1/1000,'c','Permission and energy','Energy change\n(MWh/year)')]
    for col,(step,metric,scale,letter,title,ylabel) in enumerate(panels):
        ax=fig.add_subplot(gs[1,col]);p=perm[perm.permission_step==step]
        for j,c in enumerate(('personal_center','personal_range')):
            g=p[p.control==c];x=np.full(len(g),j)+rng.normal(0,.05,len(g));ax.scatter(x,g[metric]*scale,s=22,color=CONTROL_COLORS[c],alpha=.76,edgecolor='white',lw=.35)
            ax.plot([j-.12,j+.12],[np.median(g[metric]*scale)]*2,color='black',lw=1.3)
        ax.set_xticks([0,1],['Personal\ncentre','Personal\nrange']);ax.set_ylabel(ylabel,labelpad=1);ax.axhline(0,color='#999',lw=.8);final_title(ax,letter,title);final_style(ax)
    panels=[('delta_modeled_interval_person_hour_coverage',100,'d','Placement and service','Coverage change\n(percentage points)'),('delta_annual_site_kwh',1/1000,'e','Placement and energy','Energy change\n(MWh/year)')]
    for col,(metric,scale,letter,title,ylabel) in enumerate(panels):
        ax=fig.add_subplot(gs[2,col])
        for j,c in enumerate(('personal_center','personal_range')):
            g=place[place.control==c];x=np.full(len(g),j)+rng.normal(0,.05,len(g));ax.scatter(x,g[metric]*scale,s=22,color=CONTROL_COLORS[c],alpha=.76,edgecolor='white',lw=.35)
            ax.plot([j-.12,j+.12],[np.median(g[metric]*scale)]*2,color='black',lw=1.3)
        ax.set_xticks([0,1],['Personal\ncentre','Personal\nrange']);ax.set_ylabel(ylabel,labelpad=1);ax.axhline(0,color='#999',lw=.8);final_title(ax,letter,title);final_style(ax)
    save_plot(fig,'Figure_04',size_mm=(180,188))
draw_figure_04()


In [ ]:

# Figure_05: common-baseline information audit and controller-own response.
def draw_figure_05():
    layouts=['random_mixing','demand_ordered']
    common=pd.read_csv(RESULTS/'final_submission/evidence/FINAL_COMMON_BASELINE_DR.csv')
    own=pd.read_csv(RESULTS/'final_submission/evidence/FINAL_OWN_BASELINE_DR.csv')
    pairs=pd.read_csv(RESULTS/'final_submission/evidence/FINAL_OWN_BASELINE_DR_COMPARISONS.csv')
    common=common[(common.quality_scope=='primary')&common.layout.isin(layouts)]
    own=own[(own.quality_scope=='primary')&own.layout.isin(layouts)]
    pairs=pairs[(pairs.quality_scope=='primary')&pairs.layout.isin(layouts)]
    q=own.qualified_positive_response.astype(str).str.lower().eq('true')
    fig=plt.figure(figsize=(180/25.4,190/25.4));gs=fig.add_gridspec(3,2,height_ratios=[.82,1,1],hspace=.82,wspace=.52)
    ax=fig.add_subplot(gs[0,:]);ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis('off');final_title(ax,'a','Response value includes the event and subsequent recovery')
    def timeline(y,label,stages,total):
        ax.text(.01,y+.09,label,ha='left',va='center',fontweight='bold',fontsize=9.5)
        widths={'Precondition':.14,'Response':.12,'Recovery':.29}
        colors={'Precondition':'#E8E0D0','Response':'#E6D0C4','Recovery':'#D9E7E1'}
        x=.205
        for stage,hours in stages:
            w=widths[stage]
            ax.add_patch(FancyBboxPatch((x,y),w,.18,boxstyle='round,pad=.006',fc=colors[stage],ec='#888888',lw=.8))
            ax.text(x+w/2,y+.105,stage,ha='center',va='center',fontweight='bold',fontsize=8.8)
            ax.text(x+w/2,y+.035,hours,ha='center',va='center',fontsize=8.5)
            x+=w+.015
        ax.annotate('',xy=(x+.075,y+.09),xytext=(x-.008,y+.09),arrowprops=dict(arrowstyle='->',color='#777777'))
        ax.text(x+.085,y+.09,total,ha='left',va='center',fontsize=9)
    timeline(.53,'Common baseline',[('Precondition','2 h'),('Response','2 h'),('Recovery','48 h')],'52 h accounting')
    timeline(.19,'Own baseline',[('Response','2 h'),('Recovery','48 h')],'50 h accounting')

    ax=fig.add_subplot(gs[1,0]);final_title(ax,'b','Information changes\nusable response')
    levels=['building_mean','zone_mean']; selected=[];usable=[]
    for level in levels:
        group=common[common.information_level.eq(level)]
        selected.append(int(group.selected_response_kw.gt(1e-9).sum()));usable.append(int(group.full_reviewed_usable_response_kw.gt(1e-9).sum()))
    full=common.drop_duplicates(['data_source','context','composition_id','layout','permission','event_month'])
    selected.append(int(full.ideal_full_information_response_kw.gt(1e-9).sum()));usable.append(selected[-1])
    xx=np.arange(3);ax.bar(xx-.18,selected,.36,color='#C98B77',label='Selected positive');ax.bar(xx+.18,usable,.36,color='#6F9B86',label='Usable after full review')
    ax.set_xticks(xx,['Building\nmean','Zone\nmean','Full\ndistribution']);ax.set_ylabel('Conditions / 192',labelpad=1);ax.legend(frameon=False,loc='upper right',fontsize=8.5);final_style(ax)
    ax.set_ylim(0,max(selected)*1.22)
    for i,(a,b) in enumerate(zip(selected,usable)):ax.text(i-.18,a+3,str(a),ha='center',fontsize=9);ax.text(i+.18,b+3,str(b),ha='center',fontsize=9)

    ax=fig.add_subplot(gs[1,1]);final_title(ax,'c','Response by\noperating baseline')
    own=own.assign(qualified=q);counts=own.groupby(['data_source','control']).qualified.sum().unstack(0).reindex(CONTROL_ORDER).fillna(0)
    bottom=np.zeros(3)
    for source,color in [('ASHRAE','#7FA7BA'),('CTCD','#C98B77')]:
        vals=counts.get(source,pd.Series(0,index=counts.index)).values;ax.bar(range(3),vals,bottom=bottom,color=color,label=source,width=.62);bottom+=vals
    ax.set_xticks(range(3),['Uniform\naverage','Personal\ncentre','Personal\nrange']);ax.set_ylabel('Qualified positive\nconditions / 288',labelpad=1);ax.legend(frameon=False,fontsize=8.5);final_style(ax)
    for i,v in enumerate(bottom): ax.text(i,v+.8,str(int(v)),ha='center',fontsize=9)

    ax=fig.add_subplot(gs[2,0]);final_title(ax,'d','Demand-guided minus\naverage response')
    both=pairs[(pairs.uniform_status=='selected')&(pairs.demand_guided_status=='selected')]
    rng=np.random.default_rng(9)
    for j,c in enumerate(('personal_center','personal_range')):
        g=both[both.control==c];x=np.full(len(g),j)+rng.normal(0,.045,len(g));ax.scatter(x,g.response_difference_kw,s=23,color=CONTROL_COLORS[c],alpha=.76,edgecolor='white',lw=.35);ax.plot([j-.12,j+.12],[g.response_difference_kw.median()]*2,color='black',lw=1.3)
    ax.set_xticks([0,1],['Personal\ncentre','Personal\nrange']);ax.set_ylabel('Demand-guided minus\nuniform response (kW)',labelpad=1);ax.axhline(0,color='#999',lw=.8);final_style(ax)
    ax.text(.98,.04,'51 lower; 2 higher',transform=ax.transAxes,ha='right',fontweight='bold',fontsize=9)

    ax=fig.add_subplot(gs[2,1]);final_title(ax,'e','Event saving and\n48-h recovery')
    selected=own[q];ax.scatter(selected.event_electric_saving_kwh,selected.recovery_extra_electric_kwh,c=[CONTROL_COLORS[c] for c in selected.control],s=24,alpha=.72,edgecolor='white',lw=.3)
    lim=max(selected.event_electric_saving_kwh.max(),selected.recovery_extra_electric_kwh.max())*1.04;ax.plot([0,lim],[0,lim],ls='--',color='#888',lw=1);ax.set_xlim(0,lim);ax.set_ylim(0,lim);ax.set_xlabel('Event electricity saving (kWh)');ax.set_ylabel('48-h recovery electricity (kWh)',labelpad=1);final_style(ax)
    ax.text(.97,.05,'Median recovery/event = 61.8%',transform=ax.transAxes,ha='right',fontweight='bold',fontsize=9)
    save_plot(fig,'Figure_05',size_mm=(180,190))
draw_figure_05()


In [ ]:

# Figure_06: energy bills, implementation resources and break-even value.
def draw_figure_06():
    boundaries=pd.read_csv(RESULTS/"final_submission/costs/bill_implementation_boundaries.csv")
    scenarios=pd.read_csv(RESULTS/"final_submission/costs/implementation_cost_scenarios.csv")
    costs=pd.read_csv(RESULTS/"final_submission/costs/all_signed_paired_costs.csv",low_memory=False)
    us=["Miami","PortAngeles_small2019","Seattle_small2019"]
    direct=boundaries[(boundaries.status=="matched")&(boundaries.comparison_mode=="direct")&boundaries.climate_context.isin(us)&boundaries.layout.isin(["random_mixing","demand_ordered"])]
    same=boundaries[(boundaries.status=="matched")&(boundaries.comparison_mode=="equal_service")&(boundaries.service_fraction==1)&boundaries.climate_context.isin(us)&boundaries.layout.isin(["random_mixing","demand_ordered"])]
    fig,axs=plt.subplots(3,2,figsize=(180/25.4,194/25.4));plt.subplots_adjust(hspace=.79,wspace=.50)
    rng=np.random.default_rng(4)
    ax=axs[0,0];final_title(ax,"a","Bill change when service also changes")
    for j,c in enumerate(("personal_center","personal_range")):
        g=direct[direct.control==c];x=np.full(len(g),j)+rng.normal(0,.045,len(g));ax.scatter(x,g.delta_energy_charge_usd_year,s=22,color=CONTROL_COLORS[c],alpha=.75,edgecolor="white",lw=.3);ax.plot([j-.12,j+.12],[g.delta_energy_charge_usd_year.median()]*2,color="black",lw=1.3)
        ax.set_xticks([0,1],["Personal\ncentre","Personal\nrange"]);ax.set_ylabel("Energy-bill change\n(US$/year)",labelpad=3);ax.axhline(0,color="#999",lw=.8);final_style(ax)
    ax=axs[0,1];final_title(ax,"b","Bill change under common\nannual limits")
    for j,c in enumerate(("personal_center","personal_range")):
        g=same[same.control==c];x=np.full(len(g),j)+rng.normal(0,.045,len(g));ax.scatter(x,g.delta_energy_charge_usd_year,s=19,color=CONTROL_COLORS[c],alpha=.68,edgecolor="white",lw=.3);ax.plot([j-.12,j+.12],[g.delta_energy_charge_usd_year.median()]*2,color="black",lw=1.3)
        ax.set_xticks([0,1],["Personal\ncentre","Personal\nrange"]);ax.set_ylabel("Energy-bill change\n(US$/year)",labelpad=3);ax.axhline(0,color="#999",lw=.8);final_style(ax)
    ax=axs[1,0];final_title(ax,"c","Implementation resources")
    named=scenarios[(scenarios.kind=="named_package")&(scenarios.wage_profile=="median")].copy();named=named[named.name.isin(["existing_bms_light","existing_bms_feedback","supported_feedback","optional_interface"])]
    named=named.set_index("name").reindex(["existing_bms_light","existing_bms_feedback","supported_feedback","optional_interface"])
    ax.barh(range(len(named)),named.annual_resource_cost_usd,color=["#7FA7BA","#8CB2A1","#A58AA6","#C98B77"]);ax.set_yticks(range(len(named)),["Existing","Feedback","Supported","Interface"]);ax.set_xlabel("Annualized resources (US$/year)");final_style(ax)
    ax=axs[1,1];final_title(ax,"d","Value needed after energy bills")
    cc=costs[(costs.status=="matched")&(costs.comparison_mode=="direct")&costs.climate_context.isin(us)&costs.layout.isin(["random_mixing","demand_ordered"])&costs.scenario_id.isin(["existing_bms_light__median","supported_feedback__median","optional_interface__median"])]
    labels=["Existing","Supported","Interface"];ids=["existing_bms_light__median","supported_feedback__median","optional_interface__median"]
    for i,sid in enumerate(ids):
        g=cc[cc.scenario_id==sid];vals=g.minimum_additional_annual_external_value_for_resource_breakeven_usd;ax.boxplot(vals,positions=[i],widths=.45,showfliers=False,patch_artist=True,boxprops=dict(facecolor=["#7FA7BA","#A58AA6","#C98B77"][i],alpha=.55),medianprops=dict(color="black"))
    ax.set_xticks(range(3),labels);ax.set_ylabel("External value needed\n(US$/year)",labelpad=3);final_style(ax)
    ax=axs[2,0];final_title(ax,"e","Command scope and management")
    scope=pd.read_csv(RESULTS/"final_submission/costs/command_scope_management_boundaries.csv",low_memory=False);scope=scope[(scope.status=="matched")&scope.climate_context.isin(us)&(scope.scenario_id=="existing_bms_light__median")]
    ax.scatter(scope.extra_groups_over_uniform,scope.scope_annual_hour_cost_slope_usd_per_group_hour,s=18,c="#6F9B86",alpha=.35);ax.set_xlabel("Additional command groups");ax.set_ylabel("Annual cost from one added hour\nper command group (US$/year)",labelpad=3);final_style(ax)
    ax=axs[2,1];final_title(ax,"f","Break-even value distribution")
    for sid,label,color in zip(ids,labels,["#7FA7BA","#A58AA6","#C98B77"]):
        vals=np.sort(cc[cc.scenario_id==sid].minimum_additional_annual_external_value_for_resource_breakeven_usd.dropna());y=np.arange(1,len(vals)+1)/len(vals)*100;ax.step(vals,y,where="post",label=label,color=color,lw=1.8)
    ax.set_xlabel("Required annual value (US$)");ax.set_ylabel("Cases at or below (%)");ax.legend(frameon=False);final_style(ax)
    save_plot(fig,"Figure_06",size_mm=(180,194))
draw_figure_06()


In [ ]:
def draw_s01():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S01"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    variance = panel("Fig1h.csv")

    components = list(variance.component.drop_duplicates())

    fig, a = canvas(
        [
            "Record-equal weighting",
            "Unit-equal weighting",
            "Building-equal weighting",
            "Source-equal weighting",
            "Identified-unit check",
            "Variance components",
        ]
    )

    labels = {
        "between_country": "Country",
        "between_city_within_country": "City",
        "between_building_within_city": "Building",
        "between_analysis_unit_within_building": "Within\nbuilding",
        "within_analysis_unit": "Within-unit\nvariation",
    }

    for ax, (weight, d) in zip(a, variance.groupby("weighting", sort=True)):
        # The title follows the actual group to avoid relying on sort order.
        ax.set_title(
            f"({chr(97+a.index(ax))}) "
            + weight.replace("_equal", " equal").replace("_", " ").capitalize(),
            loc="left",
            pad=18,
        )
        for j, g in enumerate(sorted(d.griffiths.unique())):
            q = d[d.griffiths == g].set_index("component").reindex(components)
            y = np.arange(len(q)) + (j - 1) * 0.22
            center = q.loo_median_share.to_numpy() * 100
            ax.errorbar(
                center,
                y,
                xerr=[
                    center - q.loo_min_share.to_numpy() * 100,
                    q.loo_max_share.to_numpy() * 100 - center,
                ],
                fmt="o",
                ms=3,
                capsize=2,
                color=GRIFFITHS_COLORS[g],
                label=f"g = {g:g}",
            )
        ax.set_yticks(
            range(len(components)),
            [labels.get(x, x.replace("_", " ")) for x in components],
            fontsize=11,
        )
        ax.set_xlabel("Archive variance share (%)")
        ax.set_xlim(-2, 85)

    a[0].legend(frameon=False, fontsize=11, loc="upper right")

    identity_variance = pd.read_csv(args.science_audit / "variance_recheck.csv")

    comps = list(identity_variance.component.drop_duplicates())

    groups = list(identity_variance.cohort.drop_duplicates())

    y = np.arange(len(groups))

    left = np.zeros(len(groups))

    for j, comp in enumerate(comps):
        q = identity_variance[identity_variance.component == comp].set_index("cohort").reindex(groups)
        a[4].barh(
            y, q.share * 100, left=left, color=VARIANCE_COLORS[j], label=comp.replace("_", " ")
        )
        left += q.share.to_numpy() * 100

    a[4].set_yticks(y, ["All records", "Identified\nsubjects"])

    a[4].set_xlabel("Archive variance share (%)")

    a[4].set_ylim(-0.5, 2.45)

    component_labels = ["Country", "City", "Building", "Within\nbuilding", "Within-unit\nvariation"]

    handles = [Patch(facecolor=VARIANCE_COLORS[j], edgecolor="none") for j in range(len(comps))]

    a[4].legend(
        handles, [label.replace("\n", " ") for label in component_labels], frameon=False, loc="upper left", ncol=2,
        handlelength=1, columnspacing=0.7, labelspacing=0.2,
    )

    arrays = [
        identity_variance[identity_variance.cohort == group].set_index("component").reindex(comps).share.to_numpy() * 100
        for group in groups
    ]

    for y, (v, w) in enumerate(zip(*arrays)):
        a[5].plot([v, w], [y, y], color="#aaaaaa", lw=1)

    for values, color, label in zip(
        arrays, [PALETTE["blue"], PALETTE["vermillion"]], ["All records", "Identified subjects"]
    ):
        a[5].scatter(values, range(5), s=35, color=color, label=label, zorder=3)

    a[5].set_yticks(range(5), component_labels)

    a[5].invert_yaxis()

    a[5].set_xlim(0, 65)

    a[5].set_xlabel("Variance share (%)")

    a[5].legend(frameon=False, loc="upper right")

    manifest.append(
        save(
            fig,
            out,
            2,
            "Figure S2 | Hierarchical variance and identity sensitivity. a–d, Leave-one-source variance shares under the four weighting schemes identified in the panel titles. Points are medians across twenty-seven source removals and whiskers give the minimum and maximum. Blue, gold and green denote Griffiths coefficients 0.33, 0.50 and 0.67 °C−1. The five components are country, city within country, building within city, analysis unit within building and repeated measurement within analysis unit. Building labels denote database groups; 150 identifiers are original and 37 inferred. e,f, Hierarchical variance shares for all canonical records and the subset with subject-within-building identifiers. Component colours in e match Figure 1d. Blue and terracotta in f identify all records and identified subjects, respectively. The identified-subject subset uses only original-source building identifiers.",
            ["Fig1h.csv", "science_audit/variance_recheck.csv"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S02"
    final_stem = "Figure_S01"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S01.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s01()

In [ ]:
def draw_s02():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S02"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    human = read("human/human_profiles_minimized.csv.gz")

    fig, a = canvas(["Uniform 24 °C", "Uniform 25 °C", "Uniform 26 °C", "Climate-group contrast"])

    summary = []

    for ax, target in zip(a[:3], [24, 25, 26]):
        for j, (g, col) in enumerate(
            zip(["reference_g033_c", "reference_g050_c", "reference_g067_c"], G_COLORS)
        ):
            values = human[g].to_numpy() - target
            ax.hist(
                values,
                bins=np.arange(-12, 12.5, 0.5),
                weights=human.unit_equal_weight,
                histtype="step",
                density=True,
                color=col,
                label=["g = 0.33", "g = 0.50", "g = 0.67"][j],
            )
            summary.append(
                {
                    "target_c": target,
                    "g_column": g,
                    "record_count": len(values),
                    "mean_absolute_gap_c": np.average(
                        np.abs(values), weights=human.unit_equal_weight
                    ),
                }
            )
        ax.axvline(0, color="#555555", lw=0.7)
        ax.set_xlim(-10, 10)
        ax.set_xlabel("Demand centre − target (°C)")
        ax.set_ylabel("Weighted density")

    a[0].legend(frameon=False, fontsize=11, loc="upper left", handlelength=.8, handletextpad=.5)

    groups = human.groupby("climate_group").size().sort_values(ascending=False).head(6).index

    vals = [human.loc[human.climate_group == g, "reference_g050_c"].values for g in groups]

    box(
        a[3],
        vals,
        list(range(1, len(groups) + 1)),
        "Demand centre (°C)",
        colors=[PALETTE[k] for k in ["blue", "vermillion", "green", "purple", "sky", "orange"]],
    )

    a[3].set_xlabel("Climate label (see caption)")

    a[3].tick_params(axis="x", labelsize=11)

    pd.DataFrame(summary).to_csv(out / "baseline_coordinate_summary.csv", index=False)

    manifest.append(
        save(
            fig,
            out,
            1,
            "Figure S1 | Reconstructed demand relative to uniform thermal coordinates. a–c, Analysis-unit-weighted demand-centre-minus-target distributions for 24, 25 and 26 °C coordinates at Griffiths coefficients 0.33, 0.50 and 0.67 °C−1, shown in blue, gold and green. d, Record-level demand-centre distributions for the six most represented climate labels, numbered in order: "
            + "; ".join(f"{i+1}, {g}" for i, g in enumerate(groups))
            + ". Boxes show medians and interquartile ranges, whiskers extend to 1.5 interquartile ranges, and the underlying data retain outliers.",
            ["human/human_profiles_minimized.csv.gz"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S01"
    final_stem = "Figure_S02"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S02.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s02()


In [ ]:
def draw_s03():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S03"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    from matplotlib.path import Path as MplPath

    from matplotlib.patches import Polygon

    land = json.loads(args.land.read_text(encoding="utf-8"))

    polygons = []

    for feature in land["features"]:
        geom = feature["geometry"]
        shapes = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
        polygons.extend(np.asarray(shape[0], dtype=float) for shape in shapes)

    lon, lat = np.meshgrid(np.arange(-179.5, 180, 1), np.arange(-54.5, 80, 1))

    xy = np.column_stack([lon.ravel(), lat.ravel()])

    land_mask = np.zeros(len(xy), dtype=bool)

    for poly in polygons:
        land_mask |= MplPath(poly).contains_points(xy)

    def distance(points, observed):
        lp = np.radians(points)
        lo = np.radians(observed)
        aa = (
            np.sin((lp[:, None, 1] - lo[None, :, 1]) / 2) ** 2
            + np.cos(lp[:, None, 1])
            * np.cos(lo[None, :, 1])
            * np.sin((lp[:, None, 0] - lo[None, :, 0]) / 2) ** 2
        )
        return 6371 * 2 * np.arcsin(np.sqrt(np.clip(aa, 0, 1)))

    def interpolate(dist, values):
        idx = np.argsort(dist, axis=1)[:, :4]
        near = np.take_along_axis(dist, idx, axis=1)
        weights = 1 / np.maximum(near, 1) ** 2
        return (weights * values[idx]).sum(axis=1) / weights.sum(axis=1)

    fig = plt.figure(figsize=(180 / 25.4, 220 / 25.4))

    gs = fig.add_gridspec(
        3,
        2,
        left=0.13,
        right=0.97,
        top=0.90,
        bottom=0.10,
        hspace=0.55,
        wspace=0.5,
        height_ratios=[1, 1, 1.1],
    )

    mapaxes = [fig.add_subplot(gs[0, :]), fig.add_subplot(gs[1, :])]

    cv = []

    grid_summaries = []

    for ax, season, label in zip(
        mapaxes,
        ["warm", "cold"],
        ["(a) Warm-season interpolation", "(b) Cold-season interpolation"],
    ):
        d = read("curves/city_season_demand_support.csv")
        d = d[(d.season_group == season) & d.supported]
        supported_count = len(d)
        d = d.dropna(subset=["map_longitude_0p5deg", "map_latitude_0p5deg", "g050_p50_c"]).copy()
        coords = d[["map_longitude_0p5deg", "map_latitude_0p5deg"]].to_numpy()
        values = d.g050_p50_c.to_numpy()
        record_counts = d.record_count.to_numpy(dtype=float)
        marker_sizes = 14 + 34 * np.sqrt(record_counts / max(record_counts.max(), 1))
        dist = distance(xy, coords)
        nearest = dist.min(axis=1)
        z = interpolate(dist, values)
        keep = land_mask & (nearest <= 1000)
        shown = np.where(keep, z, np.nan).reshape(lon.shape)
        for poly in polygons:
            ax.add_patch(Polygon(poly, facecolor="#efefef", edgecolor="#bdbdbd", lw=0.35))
        im = ax.pcolormesh(
            lon,
            lat,
            shown,
            cmap=DEMAND_CMAP,
            vmin=19.5,
            vmax=30.6,
            shading="nearest",
            rasterized=True,
        )
        ax.scatter(
            coords[:, 0],
            coords[:, 1],
            c=values,
            cmap=DEMAND_CMAP,
            vmin=19.5,
            vmax=30.6,
            s=marker_sizes,
            edgecolor="#7F7F7F",
            linewidth=0.4,
        )
        ax.set_xlim(-180, 180)
        ax.set_ylim(-55, 80)
        ax.set_aspect("equal")
        ax.set_title(label, loc="left", fontsize=12, weight="bold", pad=8)
        ax.set_xticks([-120, -60, 0, 60, 120])
        ax.set_yticks([-30, 0, 30, 60])
        ax.set_ylabel("Latitude (°)")
        ax.text(
            0.99,
            0.02,
            f"{len(d)} geolocated cities; 1,000-km mask",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=11,
        )
        cvdist = distance(coords, coords)
        np.fill_diagonal(cvdist, np.inf)
        pred = interpolate(cvdist, values)
        near = cvdist.min(axis=1)
        for city, obs, pr, km in zip(d.city, values, pred, near):
            cv.append(
                {
                    "city": city,
                    "season": season,
                    "observed_c": obs,
                    "heldout_prediction_c": pr,
                    "absolute_error_c": abs(pr - obs),
                    "nearest_training_city_km": km,
                }
            )
        for km in [500, 1000, 1500]:
            grid_summaries.append(
                {
                    "season": season,
                    "supported_city_summaries": supported_count,
                    "geolocated_city_summaries": len(d),
                    "excluded_without_scientific_coordinates": supported_count - len(d),
                    "mask_radius_km": km,
                    "supported_land_cells": int((land_mask & (nearest <= km)).sum()),
                    "land_cells": int(land_mask.sum()),
                }
            )

    cax = fig.add_axes([0.60, 0.945, 0.34, 0.012])

    bar = fig.colorbar(im, cax=cax, orientation="horizontal")

    bar.set_label("Reconstructed demand centre (°C)", labelpad=1)

    cax.xaxis.set_label_position("top")

    # Circle area conveys the number of records contributing to the city-season median.
    legend_counts = [50, 500, 4000]
    legend_sizes = [14 + 34 * np.sqrt(v / 4000) for v in legend_counts]
    fig.legend(
        [Line2D([], [], marker="o", linestyle="none", markersize=np.sqrt(v),
                markerfacecolor="white", markeredgecolor="#7F7F7F", markeredgewidth=.6)
         for v in legend_sizes],
        [f"{v:,} records" for v in legend_counts],
        loc="upper left", bbox_to_anchor=(.12, .992), frameon=False, ncol=3,
        handlelength=.8, handletextpad=.35, columnspacing=.8, borderaxespad=0,
    )

    ax = fig.add_subplot(gs[2, 0])

    ay = fig.add_subplot(gs[2, 1])

    cv = pd.DataFrame(cv)

    for j, (season, d) in enumerate(cv.groupby("season")):
        ax.scatter(
            d.observed_c,
            d.heldout_prediction_c,
            s=20,
            color=SEASON_COLORS[season],
            alpha=0.8,
            label=season.title(),
        )
        ay.scatter(
            d.nearest_training_city_km,
            d.absolute_error_c,
            s=20,
            color=SEASON_COLORS[season],
            alpha=0.8,
        )

    ax.plot([18, 32], [18, 32], ls="--", color="gray", lw=0.8)

    ax.set_xlabel("Observed city median (°C)")

    ax.set_ylabel("Held-out prediction (°C)")

    ax.legend(frameon=False, loc="upper left")

    ay.set_xlabel("Nearest training city (km)")

    ay.set_ylabel("Absolute held-out error (°C)")

    ay.axvline(1000, ls="--", color="gray", lw=0.8)

    ax.set_title("(c) Leave-one-city-out", loc="left", fontsize=12, weight="bold", pad=14)

    ay.set_title("(d) Transfer distance and error", loc="left", fontsize=12, weight="bold", pad=14)

    cv.to_csv(out / "spatial_interpolation_city_holdout.csv", index=False)

    pd.DataFrame(grid_summaries).to_csv(
        out / "spatial_interpolation_support_masks.csv", index=False
    )

    manifest.append(
        save(
            fig,
            out,
            14,
            "Figure S14 | Support-masked interpolation and out-of-city diagnostics. a,b, Four-neighbour inverse-distance-squared interpolation of observed warm- and cold-season city medians at Griffiths coefficient 0.50 °C−1. The common colour scale matches Figure 1a,b. Coloured land lies within 1,000 km of a geolocated observation. Grey land lies outside this mask. Points mark thirty warm-season and twenty-five cold-season geolocated city summaries. One supported Bangalore summary per season has a display-only coordinate and is excluded. c, Leave-one-city-out predictions without the held-out city in training. d, Absolute prediction error versus nearest training-city distance. Blue/terracotta denote cold/warm seasons. Data include errors and support masks at 500, 1,000 and 1,500 km.",
            ["curves/city_season_demand_support.csv", "natural_earth_110m_land"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S14"
    final_stem = "Figure_S03"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    caption = (
        "Supplementary Figure S3 | Geographic demand patterns and transfer diagnostics. "
        "a,b, Four-neighbour inverse-distance-squared interpolation of warm- and cold-season city median demand centres at Griffiths coefficient 0.50 °C−1. "
        "Coloured land lies within 1,000 km of a geolocated observation; grey land lies outside this support mask. Circle colour gives the city median demand centre and circle area gives record count. "
        "c, Leave-one-city-out predicted versus observed city medians. d, Absolute held-out error versus distance to the nearest training city; blue and terracotta denote cold and warm seasons."
    )
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s03()


In [ ]:
def draw_s04():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S04"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    audit = args.spatial_audit

    metrics = pd.read_csv(audit / "metrics_summary.csv")

    oof = pd.read_csv(audit / "oof_predictions.csv")

    paired = pd.read_csv(audit / "paired_cluster_bootstrap_vs_mean.csv")

    fig, a = canvas(
        [
            "Out-of-city RMSE",
            "Out-of-city MAE",
            "GPR interval calibration",
            "Coefficient sensitivity",
        ]
    )

    primary = metrics[metrics.target.eq("g050_p50_c")].copy()

    methods = ["global_training_mean", "idw2_4nn", "gpr_spherical_xyz"]

    method_labels = ["Training mean", "4-neighbour IDW²", "Spherical GPR"]

    scopes = ["all", "cold", "warm"]

    scope_labels = ["Pooled", "Cold", "Warm"]

    x = np.arange(len(scopes))

    width = 0.23

    for ax, metric, ylabel in [(a[0], "rmse_c", "RMSE (°C)"), (a[1], "mae_c", "MAE (°C)")]:
        for j, (method, label) in enumerate(zip(methods, method_labels)):
            q = primary[primary.model.eq(method)].set_index("season_scope").reindex(scopes)
            ax.bar(
                x + (j - 1) * width, q[metric], width, color=(COLORS + ["#777777"])[j], label=label
            )
        ax.set_xticks(x, scope_labels)
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, 2.8)

    a[1].legend(frameon=False, fontsize=11, loc="upper left")

    gp = oof[(oof.target == "g050_p50_c") & (oof.model == "gpr_spherical_xyz")].copy()

    covered = gp.covered_95.astype(str).str.lower().eq("true")

    for mask, color, label in [(covered, COLORS[0], "Covered"), (~covered, COLORS[1], "Missed")]:
        q = gp[mask]
        a[2].vlines(q.observed_c, q.lower_95_c, q.upper_95_c, color=color, alpha=0.55, lw=0.7)
        a[2].scatter(
            q.observed_c, q.prediction_c, s=19, color=color, alpha=0.8, label=label, zorder=3
        )

    lo = min(gp.observed_c.min(), gp.lower_95_c.min())

    hi = max(gp.observed_c.max(), gp.upper_95_c.max())

    a[2].plot([lo, hi], [lo, hi], ls="--", lw=0.8, color="gray")

    a[2].set_xlim(lo - 0.3, hi + 0.3)

    a[2].set_ylim(lo - 0.3, hi + 0.3)

    a[2].set_xlabel("Observed city median (°C)")

    a[2].set_ylabel("GPR prediction and 95% interval (°C)")

    coverage = covered.mean() * 100

    wilson_low = (
        primary[
            (primary.model == "gpr_spherical_xyz") & (primary.season_scope == "all")
        ].coverage_95_wilson_low.iloc[0]
        * 100
    )

    wilson_high = (
        primary[
            (primary.model == "gpr_spherical_xyz") & (primary.season_scope == "all")
        ].coverage_95_wilson_high.iloc[0]
        * 100
    )

    a[2].text(
        0.03,
        0.96,
        f"{covered.sum()}/{len(gp)} covered ({coverage:.1f}%)\nWilson 95%: {wilson_low:.1f}–{wilson_high:.1f}%",
        transform=a[2].transAxes,
        va="top",
    )

    a[2].legend(frameon=False, loc="lower right")

    pooled = paired[
        (paired.season_scope == "all") & paired.model.isin(["idw2_4nn", "gpr_spherical_xyz"])
    ].copy()

    target_order = ["g033_p50_c", "g050_p50_c", "g067_p50_c"]

    rows = []

    for target in target_order:
        for method in ["idw2_4nn", "gpr_spherical_xyz"]:
            rows.append(pooled[(pooled.target == target) & (pooled.model == method)].iloc[0])

    y = np.arange(len(rows))

    labels = [
        f'g = {r.target[1:4].replace("0", ".", 1)} ' + ("IDW²" if r.model == "idw2_4nn" else "GPR")
        for r in rows
    ]

    for offset, metric, marker, color, label in [
        (-0.10, "mae", "o", COLORS[0], "ΔMAE"),
        (0.10, "rmse", "^", COLORS[1], "ΔRMSE"),
    ]:
        center = np.array([r[f"delta_{metric}_c_model_minus_mean"] for r in rows])
        low = np.array([r[f"delta_{metric}_c_ci_low"] for r in rows])
        high = np.array([r[f"delta_{metric}_c_ci_high"] for r in rows])
        a[3].errorbar(
            center,
            y + offset,
            xerr=[center - low, high - center],
            fmt=marker,
            color=color,
            capsize=2,
            ms=4,
            label=label,
        )

    a[3].axvline(0, color="gray", lw=0.8, ls="--")

    a[3].set_yticks(y, labels)

    a[3].invert_yaxis()

    a[3].set_xlabel("Spatial model − training mean (°C)")

    a[3].set_xlim(-0.7, 1.25)

    a[3].plot(0.84, 2.15, marker="o", color=COLORS[0], ms=5)

    a[3].text(0.94, 2.15, "ΔMAE", va="center")

    a[3].plot(0.84, 2.55, marker="^", color=COLORS[1], ms=5)

    a[3].text(0.94, 2.55, "ΔRMSE", va="center")

    manifest.append(
        save(
            fig,
            out,
            15,
            "Figure S15 | Whole-city holdout audit of coordinate-based spatial prediction. a,b, Held-out root-mean-square error (RMSE) and mean absolute error (MAE) for the training mean, four-neighbour inverse-distance-squared interpolation (IDW²) and spherical Gaussian-process regression (GPR), at Griffiths coefficient 0.50 °C−1. c, GPR predictions and nominal 95% intervals for fifty-five held-out city–season summaries. Blue/terracotta mark covered/missed observations. Forty-five are covered (81.8%, Wilson interval 69.7–89.8%), with mean interval width 6.82 °C. d, Paired city-cluster bootstrap differences from the training-mean baseline in pooled MAE and RMSE under three Griffiths coefficients. Points and horizontal lines show estimates and intervals from fixed out-of-fold predictions. Negative values favour the spatial method.",
            [
                "spatial_method_audit/metrics_summary.csv",
                "spatial_method_audit/oof_predictions.csv",
                "spatial_method_audit/paired_cluster_bootstrap_vs_mean.csv",
            ],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S15"
    final_stem = "Figure_S04"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S04.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s04()

In [ ]:
def draw_s05():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S05"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    gender = pd.read_csv(args.science_audit / "demographic_city_season_gender_summary.csv")

    age = pd.read_csv(args.science_audit / "demographic_city_season_age_summary.csv")

    fig, a = canvas(
        [
            "Gender centre contrast",
            "Gender width contrast",
            "Age-band descriptions",
            "Demographic coverage",
        ]
    )

    for ax, metric in zip(a[:2], ["median", "p10_p90_width"]):
        for j, g in enumerate(["033", "050", "067"]):
            field = f"reference_center_g{g}_{metric}_c"
            q = gender.pivot(index=["country", "city", "season"], columns="gender", values=field)
            values = (q["female"] - q["male"]).dropna()
            ax.hist(
                values,
                bins=np.arange(-4, 4.5, 0.5),
                histtype="step",
                color=G_COLORS[j],
                label=["g = 0.33", "g = 0.50", "g = 0.67"][j],
            )
        ax.axvline(0, color="gray", lw=0.7)
        ax.set_ylabel("City–season settings")

    a[0].set_xlabel("Female − male median (°C)")

    a[1].set_xlabel("Female − male width (°C)")

    a[0].legend(frameon=False, fontsize=11)

    bands = [
        x
        for x in ["under_20", "20_29", "30_39", "40_49", "50_59", "60_plus"]
        if x in set(age.age_band)
    ]

    if not bands:
        bands = list(age.age_band.drop_duplicates())

    box(
        a[2],
        [age.loc[age.age_band == x, "reference_center_g050_median_c"] for x in bands],
        [x.replace("under_", "<").replace("_plus", "+").replace("_", "–") for x in bands],
        "City–season median (°C)",
        colors=[PALETTE[k] for k in ["blue", "vermillion", "green", "purple", "sky", "orange"]],
    )

    a[2].set_xlabel("Age band (years)")

    a[2].tick_params(axis="x", rotation=30)

    a[3].bar(
        ["Age", "Gender"], [10283 / 21418 * 100, 12941 / 21418 * 100], color=COLORS[:2], alpha=0.8
    )

    a[3].set_ylim(0, 100)

    a[3].set_ylabel("Office records with data (%)")

    a[3].text(0, 52, "10,283 / 21,418", ha="center", fontsize=11)

    a[3].text(1, 64, "12,941 / 21,418", ha="center", fontsize=11)

    manifest.append(
        save(
            fig,
            out,
            11,
            "Figure S11 | Demographic demand descriptions and coverage. a,b, Female-minus-male contrasts in reconstructed median demand and P10–P90 width within matched country–city–season settings, under Griffiths coefficients 0.33, 0.50 and 0.67 °C−1. c, City–season median demand-centre distributions by recorded age band. Boxes show medians and interquartile ranges, with whiskers at 1.5 interquartile ranges. d, Original age/gender field coverage among 21,418 office records. Displayed setting–group summaries contain at least twenty records and ten analysis units.",
            [
                "science_audit/demographic_city_season_gender_summary.csv",
                "science_audit/demographic_city_season_age_summary.csv",
                "science_audit/demographic_coverage.csv",
            ],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S11"
    final_stem = "Figure_S05"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S05.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s05()

In [ ]:
def draw_s06():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S06"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    votes = read("human/direct_vote_windows_sanitized.csv.gz")

    eligible = votes[votes.eligible.astype(str).str.lower() == "true"]

    fig, a = canvas(
        ["Singapore room windows", "Longitudinal room windows", "Requests within windows", "Compression gaps"]
    )

    for ax, code in zip(a[:2], ["NUS_HAAS", "ENTH"]):
        q = eligible[eligible.dataset_code == code]
        ax.scatter(
            q.full_burden,
            q.compressed_burden,
            s=22,
            color=PALETTE["blue"] if code == "NUS_HAAS" else PALETTE["orange"],
            alpha=0.85,
        )
        ax.plot([0, 1], [0, 1], ls="--", color="gray", lw=0.8)
        ax.set_xlim(-0.03, 1.03)
        ax.set_ylim(-0.03, 1.03)
        ax.set_xlabel("Full vote burden")
        ax.set_ylabel("Burden after averaging")
        ax.text(0.04, 0.94, f"n = {len(q)} windows", transform=ax.transAxes, va="top")

    d = panel("Fig2h.csv").drop_duplicates("dataset")

    xs = np.arange(len(d))

    w = 0.24

    for j, (col, label) in enumerate(
        [
            ("disagreement_count", "Differing"),
            ("opposing_count", "Opposing"),
            ("cancellation_count", "Cancelled"),
        ]
    ):
        a[2].bar(
            xs + (j - 1) * w,
            d[col],
            w,
            label=label,
            color=[PALETTE["blue"], PALETTE["vermillion"], PALETTE["dark_grey"]][j],
        )

    a[2].set_xticks(xs, d.dataset.replace({"NUS_HAAS": "Singapore", "ENTH": "Longitudinal"}))

    a[2].set_ylabel("Windows")

    a[2].legend(frameon=False, fontsize=11)

    gap_values = [
        eligible.loc[eligible.dataset_code == x, "jensen_gap"].values for x in ["NUS_HAAS", "ENTH"]
    ]

    box(
        a[3],
        gap_values,
        ["Singapore", "Longitudinal"],
        "Full minus averaged burden",
        colors=[PALETTE["blue"], PALETTE["orange"]],
    )

    for j, values in enumerate(gap_values):
        a[3].scatter(
            j + 1 + np.linspace(-0.15, 0.15, len(values)),
            values,
            s=10,
            color=[PALETTE["blue"], PALETTE["orange"]][j],
            alpha=0.8,
        )

    manifest.append(
        save(
            fig,
            out,
            4,
            "Figure S4 | Shared-room vote replication. a,b, Mean absolute vote (full burden) versus absolute mean vote (burden after averaging) for eligible NUS and ENTH thirty-minute room-windows. Dashed lines mark equality. c, Window counts containing differing categories, opposing warmer/cooler requests, and exact cancellation. d, Full-minus-averaged burden distributions. The National University of Singapore (NUS) and ENTH datasets contain 63/34 windows and 225/81 votes. Boxes show medians and interquartile ranges, whiskers extend to 1.5 interquartile ranges, and points show individual windows.",
            ["human/direct_vote_windows_sanitized.csv.gz", "Fig2h.csv"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S04"
    final_stem = "Figure_S06"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S06.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s06()

In [ ]:
# Figure_S07: cohort construction and simulated compositions.
human=pd.read_csv(RESULTS/"legacy_analysis/human/human_profiles_minimized.csv.gz")
intervals=pd.read_csv(RESULTS/"legacy_analysis/human/ltpref_intervals_minimized.csv.gz")
ctcd_centres=pd.read_csv(RESULTS/"CTCD_external/CTCD_REFERENCE_HISTOGRAM.csv")
ctcd_widths=pd.read_csv(RESULTS/"CTCD_external/CTCD_INTERVAL_HISTOGRAM.csv")
compositions=pd.read_csv(RESULTS/"Joint_main/DEMAND_COMPOSITION_PROFILES.csv")

source_colors={"ASHRAE":"#79A9BE","CTCD":"#C98F7A"}
composition_order=sorted(compositions.composition_id.unique())
composition_labels={cid:("A" if compositions.loc[compositions.composition_id.eq(cid),"data_source"].iloc[0]=="ASHRAE" else "C")+str(i%3+1) for i,cid in enumerate(composition_order)}
compositions["composition_label"]=compositions.composition_id.map(composition_labels)
label_order=[composition_labels[x] for x in composition_order]

fig,axes=plt.subplots(2,3,figsize=(180/25.4,132/25.4),constrained_layout=True)
# a: record flow
observed=[len(human),int(ctcd_centres.scope_records.max())]
executable=[len(intervals),int(ctcd_widths.scope_records.max())]
x=np.arange(2);w=.34
axes[0,0].bar(x-w/2,observed,w,color="#D5D3CF",label="Harmonized observations")
axes[0,0].bar(x+w/2,executable,w,color=[source_colors["ASHRAE"],source_colors["CTCD"]],label="Executable demand profiles")
for j,vals in enumerate([observed,executable]):
    for i,v in enumerate(vals):axes[0,0].text(i+(j-.5)*w,v-700,f"{v:,}",ha="center",va="top",rotation=90,fontsize=8.5)
axes[0,0].set_xticks(x,["ASHRAE","CTCD"]);axes[0,0].set_ylabel("Records or profiles");axes[0,0].set_ylim(0,24500)
axes[0,0].legend(frameon=False,fontsize=8.5,loc="lower left")
final_title(axes[0,0],"a","Cohort flow");final_style(axes[0,0])

# b: demand-centre densities
bins=np.arange(10,38.01,.5)
axes[0,1].hist(human.reference_g050_c.dropna(),bins=bins,density=True,histtype="step",lw=1.8,color=source_colors["ASHRAE"],label="ASHRAE")
q=ctcd_centres[ctcd_centres.griffiths_coefficient.eq(.50)];edges=np.r_[q.bin_lower_c,q.bin_upper_c.iloc[-1]]
axes[0,1].stairs(q.density_per_c,edges,lw=1.8,color=source_colors["CTCD"],label="CTCD")
axes[0,1].set_xlabel("Demand centre (°C)");axes[0,1].set_ylabel("Record density");axes[0,1].legend(frameon=False)
final_title(axes[0,1],"b","Demand centres");final_style(axes[0,1])

# c: Personal range widths
ash_width=(intervals.interval_upper_c-intervals.interval_lower_c).dropna();edges=np.r_[ctcd_widths.bin_lower_c,ctcd_widths.bin_upper_c.iloc[-1]]
axes[0,2].hist(ash_width,bins=edges,density=True,histtype="step",lw=1.8,color=source_colors["ASHRAE"],label="ASHRAE")
axes[0,2].stairs(ctcd_widths.density_per_c,edges,lw=1.8,color=source_colors["CTCD"],label="CTCD")
axes[0,2].set_xlabel("Personal range width (°C)");axes[0,2].set_ylabel("Profile density");axes[0,2].legend(frameon=False)
final_title(axes[0,2],"c","Personal range widths");final_style(axes[0,2])

# d-e: six fixed compositions
for panel,col,ylabel,title in [(axes[1,0],"centre_c","Demand centre (°C)","Composition centres"),(axes[1,1],"range_width_c","Personal range width (°C)","Composition ranges")]:
    for j,label in enumerate(label_order):
        q=compositions[compositions.composition_label.eq(label)];vals=q[col].dropna().to_numpy();color=source_colors[q.data_source.iloc[0]]
        parts=panel.violinplot(vals,positions=[j],widths=.78,showextrema=False,showmedians=True)
        for body in parts["bodies"]:body.set_facecolor(color);body.set_alpha(.66)
        parts["cmedians"].set_color("black")
    panel.set_xticks(range(6),label_order);panel.set_ylabel(ylabel);panel.set_xlabel("Fixed composition")
    final_title(panel,"d" if panel is axes[1,0] else "e",title);final_style(panel)

# f: joint centre-width structure
markers={label:mk for label,mk in zip(label_order,["o","s","^","o","s","^"])}
for label in label_order:
    q=compositions[compositions.composition_label.eq(label)];source=q.data_source.iloc[0]
    axes[1,2].scatter(q.centre_c,q.range_width_c,s=13,marker=markers[label],color=source_colors[source],alpha=.42,linewidths=0,label=label)
axes[1,2].set_xlabel("Demand centre (°C)");axes[1,2].set_ylabel("Personal range width (°C)")
handles=[Line2D([0],[0],marker="o",linestyle="",color=source_colors["ASHRAE"],label="ASHRAE compositions A1–A3"),Line2D([0],[0],marker="o",linestyle="",color=source_colors["CTCD"],label="CTCD compositions C1–C3")]
axes[1,2].legend(handles=handles,frameon=False,fontsize=8.5,loc="upper right")
final_title(axes[1,2],"f","Centre–range structure");final_style(axes[1,2])

save_plot(fig,"Figure_S07",size_mm=(180,132))
caption=("Supplementary Figure S7 | Cohort construction and simulated demand compositions. a, Harmonized observations and executable demand profiles from the ASHRAE and CTCD archives. "
         "b, Demand-centre distributions reconstructed with Griffiths coefficient 0.50 °C−1. c, Personal range width distributions from the common demand-reconstruction workflow. "
         "d,e, Demand centres and Personal range widths in the six fixed 267-profile compositions used in the simulations; A1–A3 are sampled from ASHRAE and C1–C3 from CTCD. "
         "f, Joint centre–width structure of all 1,602 profiles in those compositions. Violin centres mark medians.")
(PLOT/"supplementary/Figure_S07_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S08: fixed-plan information diagnostic.
qa = json.loads((RESULTS / "final_submission/evidence/FINAL_EVIDENCE_QA.json").read_text(encoding="utf-8"))
c = qa["fixed_plan_information_diagnostic"]["counts"]
phase = pd.read_csv(ROOT / "output/replay/supporting_evidence/analysis/tables/result6_phase_map.csv")
fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 120/25.4), constrained_layout=True)
ax = axes.ravel()

labels = ["Mean-safe", "Distribution-\nrequired", "Unresolved"]
values = [c["zone_mean_safe_cells"], c["distribution_required_cells"], c["finite_library_unresolved_cells"]]
colors = ["#9BBFC9", "#C98F7A", "#D5D3CF"]
ax[0].bar(labels, values, color=colors, width=.66)
ax[0].tick_params(axis="x", labelsize=9)
for x, y in enumerate(values):
    ax[0].text(x, y+16, f"{y}", ha="center", va="bottom", fontsize=10)
ax[0].set_ylabel("Design settings")
ax[0].set_ylim(0, 470)
final_title(ax[0], "a", "Decision classes")
final_style(ax[0])

resolvable = c["full_profile_resolvable_cells"]
safe = c["zone_mean_safe_cells"]
required = c["distribution_required_cells"]
ax[1].barh([0], [safe/resolvable*100], color=colors[0], label="Mean-safe")
ax[1].barh([0], [required/resolvable*100], left=[safe/resolvable*100], color=colors[1], label="Distribution-required")
ax[1].text(safe/resolvable*50, 0, f"{safe/resolvable*100:.1f}%", ha="center", va="center")
ax[1].text((safe+required/2)/resolvable*100, 0, f"{required/resolvable*100:.1f}%", ha="center", va="center")
ax[1].set_xlim(0,100); ax[1].set_yticks([]); ax[1].set_xlabel("Resolvable settings (%)")
ax[1].legend(frameon=False, loc="upper center", bbox_to_anchor=(.5,-.18), ncol=1)
final_title(ax[1], "b", "Resolvable decisions")
final_style(ax[1])

fail = c["zone_mean_selected_plan_fails_full_review_cells"]
passed = resolvable-fail
ax[2].bar(["Passed full\nreview", "Failed full\nreview"], [passed, fail], color=["#9BBFC9", "#C98F7A"], width=.68)
for x, y in enumerate([passed, fail]):
    ax[2].text(x, y+13, f"{y}\n({y/resolvable*100:.1f}%)", ha="center", va="bottom", fontsize=10)
ax[2].set_ylabel("Zone-mean selections")
ax[2].set_ylim(0,420)
final_title(ax[2], "c", "Zone-mean review")
final_style(ax[2])

resolved = phase.loc[~phase["classification"].eq("finite-library-unresolved")].copy()
resolved["selected_plan_failed"] = ~resolved["mean_only_selected_full_eligible"].astype(str).str.lower().eq("true")
by_limit = resolved.groupby("service_fraction", as_index=False)["selected_plan_failed"].agg(["mean", "size"]).reset_index()
ax[3].plot(by_limit["service_fraction"], by_limit["mean"]*100, "o-", color="#C98F7A", lw=1.8)
for row in by_limit.itertuples():
    ax[3].text(row.service_fraction, row.mean*100+2.0, f"{row.mean*100:.0f}%", ha="center", va="bottom", fontsize=10)
ax[3].set_xlabel("Service-limit fraction")
ax[3].set_ylabel("Selected plans failing full review (%)")
ax[3].set_xticks(by_limit["service_fraction"])
ax[3].set_ylim(0, max(65, float((by_limit["mean"]*100).max()+10)))
final_title(ax[3], "d", "Failure by service limit")
final_style(ax[3])

save_plot(fig, "Figure_S08", size_mm=(180,120))
caption = ("Supplementary Figure S8 | Fixed-plan information diagnostic. a, Classification of 1,008 design settings using an identical physical plan library and service limits. "
           "Mean-safe settings give the same qualification under a zone mean and the complete within-zone demand distribution; distribution-required settings change qualification; unresolved settings contain no qualifying plan in the tested library. "
           "b, Shares among the 606 settings with at least one plan qualified under full-demand review. c, Full-demand review of the plans selected from zone-mean information in those 606 settings. "
           "d, Share of zone-mean selections that fail full-demand review at each service-limit fraction; percentages use the resolvable settings at that fraction.")
(PLOT/"supplementary/Figure_S08_caption.txt").write_text(caption, encoding="utf-8")


In [ ]:

# Figure_S09: engineering qualification and robustness.
paths = {
    ("ASHRAE", "Primary"): RESULTS/"Reviewer_repair/repaired_controls_ashrae_full_analysis/ANALYSIS_QA.json",
    ("ASHRAE", "Strict"): RESULTS/"Reviewer_repair/repaired_controls_ashrae_strict_analysis/ANALYSIS_QA.json",
    ("CTCD", "Primary"): RESULTS/"Reviewer_repair/repaired_controls_ctcd_full_analysis/ANALYSIS_QA.json",
    ("CTCD", "Strict"): RESULTS/"Reviewer_repair/repaired_controls_ctcd_strict_analysis/ANALYSIS_QA.json",
}
rows=[]
for (source,scope), path in paths.items():
    q=json.loads(path.read_text(encoding="utf-8")); q.update(source=source,scope=scope); rows.append(q)
qdf=pd.DataFrame(rows)
comp=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_ANNUAL_COMPARISONS.csv")
comp=comp[(comp.comparison_mode=="direct") & (comp.permission=="native_zone") & comp.layout.isin(["random_mixing","demand_ordered"]) & (comp.status=="matched")]

fig,axes=plt.subplots(2,2,figsize=(180/25.4,132/25.4),constrained_layout=True)
x=np.arange(2); width=.23
for j,(key,label,color) in enumerate([("logical_candidates","Logical","#D5D3CF"),("unique_physical_runs","Unique","#9BBFC9"),("engineering_qualified_runs","Primary-qualified","#78A28E")]):
    vals=[qdf[(qdf.source==s)&(qdf.scope==("Primary" if key=="engineering_qualified_runs" else "Primary"))][key].iloc[0] for s in ["ASHRAE","CTCD"]]
    axes[0,0].bar(x+(j-1)*width,vals,width,color=color,label=label)
axes[0,0].set_xticks(x,["ASHRAE","CTCD"]); axes[0,0].set_ylabel("Runs or logical candidates")
axes[0,0].legend(frameon=False,ncol=1,loc="upper right")
final_title(axes[0,0],"a","Simulation support"); final_style(axes[0,0])

for j,scope in enumerate(["Primary","Strict"]):
    vals=[]
    for s in ["ASHRAE","CTCD"]:
        r=qdf[(qdf.source==s)&(qdf.scope==scope)].iloc[0]
        vals.append(r.engineering_qualified_runs/r.unique_physical_runs*100)
    axes[0,1].bar(x+(j-.5)*.32,vals,.32,color=["#78A28E","#C6A064"][j],label=scope)
axes[0,1].set_xticks(x,["ASHRAE","CTCD"]); axes[0,1].set_ylabel("Qualified physical runs (%)"); axes[0,1].set_ylim(0,100)
axes[0,1].legend(frameon=False,loc="upper right")
final_title(axes[0,1],"b","Primary and strict checks"); final_style(axes[0,1])

pair_rows=[]
for _,r in qdf.iterrows():
    pair_rows += [(r.source,r.scope,"Direct",r.direct_matched_pairs),(r.source,r.scope,"Same service",r.equal_service_matched_pairs)]
pdf=pd.DataFrame(pair_rows,columns=["source","scope","comparison","count"])
for j,scope in enumerate(["Primary","Strict"]):
    v=pdf[pdf.scope==scope].pivot(index="source",columns="comparison",values="count").reindex(["ASHRAE","CTCD"])
    axes[1,0].bar(x+(j-.5)*.32,v["Direct"],.32,color=["#9BBFC9","#C98F7A"][j],label=f"Direct, {scope.lower()}")
axes[1,0].set_xticks(x,["ASHRAE","CTCD"]); axes[1,0].set_ylabel("Matched direct comparisons")
axes[1,0].legend(frameon=False,fontsize=9)
final_title(axes[1,0],"c","Matched comparison support"); final_style(axes[1,0])

for j,scope in enumerate(["primary","strict"]):
    d=comp[comp.quality_scope==scope].delta_site_pct.dropna()
    axes[1,1].boxplot(d,positions=[j],widths=.5,patch_artist=True,showfliers=False,
                      boxprops=dict(facecolor=["#9BBFC9","#C98F7A"][j],edgecolor="#777777"),
                      medianprops=dict(color="black"),whiskerprops=dict(color="#777777"),capprops=dict(color="#777777"))
axes[1,1].axhline(0,color="#B5B5B5",lw=.8); axes[1,1].set_xticks([0,1],["Primary","Strict"]); axes[1,1].set_ylabel("Annual energy change (%)")
final_title(axes[1,1],"d","Result robustness"); final_style(axes[1,1])

save_plot(fig,"Figure_S09",size_mm=(180,132))
caption=("Supplementary Figure S9 | Engineering qualification and robustness. a, Logical candidates, unique physical simulations and runs entering the primary analysis for the two harmonized demand archives. "
         "b, Shares of unique physical simulations meeting the primary and strict engineering checks. c, Matched direct comparisons supported by each archive under the two checks. d, Annual site-energy differences for direct native-zone comparisons under the primary and strict checks; boxes show medians and interquartile ranges and whiskers extend to 1.5 interquartile ranges.")
(PLOT/"supplementary/Figure_S09_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S10: annual thermal-service endpoints.
d=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_ANNUAL_COMPARISONS.csv")
d=d[(d.quality_scope=="primary")&(d.comparison_mode=="direct")&(d.permission=="native_zone")&(d.status=="matched")&d.layout.isin(["random_mixing","demand_ordered"])].copy()
metrics=[
    ("delta_mean_reference_mismatch_c",-1,"Mean mismatch reduction (°C)"),
    ("delta_fixed_tail_reference_mismatch_c",-1,"Fixed-tail mismatch reduction (°C)"),
    ("delta_modeled_interval_person_hour_coverage",100,"Interval coverage gain (percentage points)"),
    ("delta_modeled_interval_mean_excess_c",-1,"Interval-excess reduction (°C)"),
]
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)
rng=np.random.default_rng(20261026)
for i,(col,mult,ylabel) in enumerate(metrics):
    ax=axes.flat[i]
    for j,cntl in enumerate(["personal_center","personal_range"]):
        for source,marker,dx in [("ASHRAE","o",-.09),("CTCD","s",.09)]:
            vals=d[(d.alternative==cntl)&(d.data_source==source)][col].dropna().to_numpy()*mult
            x=j+dx+rng.normal(0,.025,len(vals))
            ax.scatter(x,vals,s=22,marker=marker,color=CONTROL_COLORS[cntl],alpha=.72,edgecolor="white",linewidth=.35)
        vals=d[d.alternative==cntl][col].dropna().to_numpy()*mult
        ax.plot([j-.18,j+.18],[np.median(vals)]*2,color="black",lw=1.6)
    ax.axhline(0,color="#B5B5B5",lw=.8)
    ax.set_xticks([0,1],["Personal\ncentre","Personal\nrange"]); ax.set_ylabel(ylabel)
    final_title(ax,chr(97+i),["Population mean","Fixed reference tail","Personal range coverage","Personal range excess"][i]); final_style(ax)
handles=[Line2D([0],[0],marker="o",linestyle="",color="#777777",label="ASHRAE"),Line2D([0],[0],marker="s",linestyle="",color="#777777",label="CTCD")]
axes[0,0].legend(handles=handles,frameon=False,loc="upper right")
save_plot(fig,"Figure_S10",size_mm=(180,128))
caption=("Supplementary Figure S10 | Annual thermal-service endpoints in 69 matched direct native-zone comparisons across the two layouts available to both archives. "
         "a, Reduction in population-average reference-temperature mismatch. b, Reduction in mismatch for the fixed reference-tail profiles. c, Change in occupied person-hour coverage by Personal ranges. d, Reduction in mean temperature excess outside Personal ranges. "
         "Circles and squares denote ASHRAE and CTCD compositions; horizontal black lines are medians. Positive values denote improvement for each displayed endpoint.")
(PLOT/"supplementary/Figure_S10_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S11: annual energy, carbon and carrier composition.
d=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_ANNUAL_COMPARISONS.csv")
d=d[(d.quality_scope=="primary")&(d.comparison_mode=="direct")&(d.permission=="native_zone")&(d.status=="matched")&d.layout.isin(["random_mixing","demand_ordered"])].copy()
fig,axes=plt.subplots(2,2,figsize=(180/25.4,130/25.4),constrained_layout=True)
rng=np.random.default_rng(20261027)
for ax,col,ylabel,title in [(axes[0,0],"delta_site_pct","Annual site-energy change (%)","Annual energy"),(axes[0,1],"delta_annual_operational_co2_kg","Operational carbon change (kgCO₂/year)","Annual operational carbon")]:
    for j,cntl in enumerate(["personal_center","personal_range"]):
        vals=d[d.alternative==cntl][col].dropna().to_numpy(); x=j+rng.normal(0,.055,len(vals))
        ax.scatter(x,vals,s=22,color=CONTROL_COLORS[cntl],alpha=.72,edgecolor="white",linewidth=.35)
        ax.plot([j-.16,j+.16],[np.median(vals)]*2,color="black",lw=1.5)
    ax.set_xticks([0,1],["Personal\ncentre","Personal\nrange"]); ax.set_ylabel(ylabel); ax.axhline(0,color="#B5B5B5",lw=.8)
    final_title(ax,"a" if ax is axes[0,0] else "b",title); final_style(ax)

carrier_cols=[("delta_annual_cooling_electric_kwh","Cooling electricity","#7FAFC4"),("delta_annual_heating_electric_kwh","Heating electricity","#C98F7A"),("delta_annual_fans_electric_kwh","Fan electricity","#9A94B0"),("delta_annual_gas_hhv_kwh","Heating gas","#C6A064")]
summary=d.groupby("alternative")[[x[0] for x in carrier_cols]].median().reindex(["personal_center","personal_range"])
xcar=np.arange(len(carrier_cols)); width=.34
for j,cntl in enumerate(summary.index):
    vals=[summary.loc[cntl,col]/1000 for col,_,_ in carrier_cols]
    axes[1,0].bar(xcar+(j-.5)*width,vals,width,color=CONTROL_COLORS[cntl],label=CONTROL_LABELS[cntl])
axes[1,0].set_xticks(xcar,["Cooling\nelectricity","Heating\nelectricity","Fan\nelectricity","Heating\ngas"],fontsize=9); axes[1,0].set_ylabel("Median annual change (MWh)"); axes[1,0].axhline(0,color="#777777",lw=.8)
axes[1,0].legend(frameon=False,ncol=1,fontsize=8.5,loc="best")
final_title(axes[1,0],"c","Energy carriers and end uses"); final_style(axes[1,0])

for cntl in ["personal_center","personal_range"]:
    q=d[d.alternative==cntl]
    axes[1,1].scatter(q.delta_annual_site_kwh/1000,q.delta_annual_operational_co2_kg/1000,s=25,color=CONTROL_COLORS[cntl],alpha=.72,label=CONTROL_LABELS[cntl],edgecolor="white",linewidth=.35)
axes[1,1].axhline(0,color="#B5B5B5",lw=.8);axes[1,1].axvline(0,color="#B5B5B5",lw=.8)
axes[1,1].set_xlabel("Annual site-energy change (MWh)");axes[1,1].set_ylabel("Operational carbon change (tCO₂/year)")
axes[1,1].legend(frameon=False,fontsize=9)
final_title(axes[1,1],"d","Energy and carbon"); final_style(axes[1,1])
save_plot(fig,"Figure_S11",size_mm=(180,130))
caption=("Supplementary Figure S11 | Annual resource changes in the 69 matched direct native-zone comparisons. a,b, Changes in annual site energy and operational carbon for the two demand-guided controls relative to Uniform average. "
         "Points are matched comparisons and black lines are medians. c, Median changes in cooling electricity, heating electricity, fan electricity and heating-gas higher heating value. d, Matched annual site-energy and operational-carbon changes.")
(PLOT/"supplementary/Figure_S11_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S12: common-service qualification and energy comparisons.
s=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_ANNUAL_SELECTIONS.csv")
s=s[(s.quality_scope=="primary")&s.layout.isin(["random_mixing","demand_ordered"])].copy()
c=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_ANNUAL_COMPARISONS.csv")
c=c[(c.quality_scope=="primary")&(c.comparison_mode=="equal_service")&c.layout.isin(["random_mixing","demand_ordered"])&(c.status=="matched")].copy()
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)
fractions=[1.0,.75,.5]; x=np.arange(len(fractions)); width=.23
for j,cntl in enumerate(CONTROL_ORDER):
    vals=[len(s[(s.control==cntl)&(s.service_fraction==f)&(s.selection_status=="selected")]) for f in fractions]
    axes[0,0].bar(x+(j-1)*width,vals,width,color=CONTROL_COLORS[cntl],label=CONTROL_LABELS[cntl])
axes[0,0].set_xticks(x,["1.00","0.75","0.50"]); axes[0,0].set_xlabel("Service-limit fraction"); axes[0,0].set_ylabel("Selected conditions")
axes[0,0].legend(frameon=False,fontsize=8.5)
final_title(axes[0,0],"a","Available operating choices"); final_style(axes[0,0])

q=c[c.service_fraction==1.0]
bins=np.linspace(q.delta_site_pct.quantile(.01),q.delta_site_pct.quantile(.99),20)
for cntl in ["personal_center","personal_range"]:
    axes[0,1].hist(q[q.alternative==cntl].delta_site_pct,bins=bins,histtype="step",lw=2,color=CONTROL_COLORS[cntl],label=CONTROL_LABELS[cntl])
axes[0,1].axvline(0,color="#777777",lw=.8); axes[0,1].set_xlabel("Annual site-energy change (%)"); axes[0,1].set_ylabel("Matched comparisons")
axes[0,1].legend(frameon=False,fontsize=9)
final_title(axes[0,1],"b","Same-service energy"); final_style(axes[0,1])

for j,f in enumerate(fractions):
    vals=c[c.service_fraction==f].delta_site_pct.dropna().to_numpy()
    if len(vals):
        parts=axes[1,0].violinplot(vals,positions=[j],widths=.72,showextrema=False,showmedians=True)
        for body in parts["bodies"]: body.set_facecolor(["#9BBFC9","#C6A064","#C98F7A"][j]); body.set_alpha(.75)
        parts["cmedians"].set_color("black")
axes[1,0].axhline(0,color="#B5B5B5",lw=.8);axes[1,0].set_xticks(x,["1.00","0.75","0.50"]);axes[1,0].set_xlabel("Service-limit fraction");axes[1,0].set_ylabel("Annual site-energy change (%)")
final_title(axes[1,0],"c","Energy under tighter limits"); final_style(axes[1,0])

table=[]
for f in fractions:
    for cntl in CONTROL_ORDER:
        table.append((f,cntl,len(s[(s.service_fraction==f)&(s.control==cntl)&(s.selection_status=="selected")])))
hm=pd.DataFrame(table,columns=["fraction","control","count"]).pivot(index="control",columns="fraction",values="count").reindex(CONTROL_ORDER)[fractions]
im=axes[1,1].imshow(hm,cmap="GnBu",aspect="auto")
for i in range(3):
    for j in range(3): axes[1,1].text(j,i,str(int(hm.iloc[i,j])),ha="center",va="center",color="black")
axes[1,1].set_xticks(range(3),["1.00","0.75","0.50"]);axes[1,1].set_yticks(range(3),["Uniform average","Personal centre","Personal range"]);axes[1,1].set_xlabel("Service-limit fraction")
final_title(axes[1,1],"d","Qualification counts"); final_style(axes[1,1])
save_plot(fig,"Figure_S12",size_mm=(180,128))
caption=("Supplementary Figure S12 | Common-service comparisons. a, Number of operating conditions with a selected plan at service-limit fractions 1.00, 0.75 and 0.50, using the two layouts shared by both archives. "
         "b, Annual site-energy changes for matched demand-guided versus Uniform average selections at fraction 1.00. c, Distributions of matched energy changes as the service limit tightens. d, Selection counts by control and service-limit fraction.")
(PLOT/"supplementary/Figure_S12_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S13: seasonal reallocation.
d=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_SEASONAL_DIFFERENCES.csv")
d=d[(d.quality_scope=="primary")&(d.comparison_mode=="direct")&(d.permission=="native_zone")&d.layout.isin(["random_mixing","demand_ordered"])].copy()
key=["data_source","context","composition_id","layout","control"]
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)
seasons=["DJF","MAM","JJA","SON"]; scol={"DJF":"#7FAFC4","MAM":"#8FB89B","JJA":"#C98F7A","SON":"#C6A064"}
for j,season in enumerate(seasons):
    vals=d[d.season==season].delta_site_kwh/1000
    parts=axes[0,0].violinplot(vals.dropna(),positions=[j],widths=.72,showextrema=False,showmedians=True)
    for b in parts["bodies"]:b.set_facecolor(scol[season]);b.set_alpha(.75)
    parts["cmedians"].set_color("black")
axes[0,0].axhline(0,color="#B5B5B5",lw=.8);axes[0,0].set_xticks(range(4),seasons);axes[0,0].set_ylabel("Seasonal site-energy change (MWh)")
final_title(axes[0,0],"a","Seasonal energy");final_style(axes[0,0])

for j,season in enumerate(seasons):
    vals=d[d.season==season].delta_operational_co2_kg/1000
    axes[0,1].boxplot(vals.dropna(),positions=[j],widths=.55,patch_artist=True,showfliers=False,boxprops=dict(facecolor=scol[season],edgecolor="#777777"),medianprops=dict(color="black"),whiskerprops=dict(color="#777777"),capprops=dict(color="#777777"))
axes[0,1].axhline(0,color="#B5B5B5",lw=.8);axes[0,1].set_xticks(range(4),seasons);axes[0,1].set_ylabel("Seasonal carbon change (tCO₂)")
final_title(axes[0,1],"b","Seasonal operational carbon");final_style(axes[0,1])

shares=[]
for season in seasons:
    v=d[d.season==season].delta_site_kwh.dropna(); shares.append(((v>0).sum()/len(v)*100,(v<0).sum()/len(v)*100))
axes[1,0].bar(range(4),[x[0] for x in shares],color="#C98F7A",label="Increase")
axes[1,0].bar(range(4),[x[1] for x in shares],bottom=[x[0] for x in shares],color="#9BBFC9",label="Decrease")
axes[1,0].set_xticks(range(4),seasons);axes[1,0].set_ylabel("Matched comparisons (%)");axes[1,0].set_ylim(0,100);axes[1,0].legend(frameon=False,ncol=2)
final_title(axes[1,0],"c","Direction by season");final_style(axes[1,0])

annual=[]
for _,g in d.groupby(key):
    v=g.set_index("season").reindex(seasons).delta_site_kwh.dropna().to_numpy()
    if len(v)==4 and np.any(v>0) and np.any(v<0): annual.append(1-abs(v.sum())/np.abs(v).sum())
axes[1,1].hist(np.asarray(annual)*100,bins=np.linspace(0,100,12),color="#9A94B0",edgecolor="white")
axes[1,1].axvline(np.median(annual)*100,color="black",lw=1.5,label=f"Median {np.median(annual)*100:.1f}%")
axes[1,1].set_xlabel("Cross-season cancellation (%)");axes[1,1].set_ylabel("Mixed-sign annual comparisons");axes[1,1].legend(frameon=False)
final_title(axes[1,1],"d","Annual cancellation");final_style(axes[1,1])
save_plot(fig,"Figure_S13",size_mm=(180,128))
caption=("Supplementary Figure S13 | Seasonal reallocation in direct native-zone comparisons. a,b, Seasonal changes in site energy and operational carbon for December–February, March–May, June–August and September–November. "
         "c, Shares of seasonal comparisons with increased or decreased site energy. d, Fraction of absolute seasonal changes cancelled across seasons for annual comparisons containing both positive and negative seasonal changes; the vertical line is the median.")
(PLOT/"supplementary/Figure_S13_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S14: occupant placement effects.
d=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_PLACEMENT_EFFECTS.csv")
d=d[d.quality_scope=="primary"].copy()
metrics=[("delta_mean_reference_mismatch_c",-1,"Mean mismatch reduction (°C)"),("delta_fixed_tail_reference_mismatch_c",-1,"Fixed-tail mismatch reduction (°C)"),("delta_modeled_interval_person_hour_coverage",100,"Interval coverage gain (percentage points)"),("delta_annual_site_kwh",1/1000,"Annual site-energy change (MWh)")]
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)
rng=np.random.default_rng(20261028)
for i,(col,mult,ylabel) in enumerate(metrics):
    ax=axes.flat[i]
    for j,cntl in enumerate(CONTROL_ORDER):
        vals=d[d.control==cntl][col].dropna().to_numpy()*mult
        ax.scatter(j+rng.normal(0,.045,len(vals)),vals,s=20,color=CONTROL_COLORS[cntl],alpha=.68,edgecolor="white",linewidth=.3)
        if len(vals):ax.plot([j-.15,j+.15],[np.median(vals)]*2,color="black",lw=1.4)
    ax.axhline(0,color="#B5B5B5",lw=.8);ax.set_xticks(range(3),["Uniform\naverage","Personal\ncentre","Personal\nrange"]);ax.set_ylabel(ylabel)
    final_title(ax,chr(97+i),["Population mean","Fixed reference tail","Personal range coverage","Energy consequence"][i]);final_style(ax)
save_plot(fig,"Figure_S14",size_mm=(180,128))
caption=("Supplementary Figure S14 | Effects of changing occupant placement from random mixing to demand-ordered grouping while holding each controller and command permission fixed. "
         "a,b, Reductions in population-average and fixed-reference-tail mismatch. c, Gain in occupied person-hour Personal range coverage. d, Change in annual site energy. Points are matched placement comparisons and black lines are medians; positive values in a–c denote service improvement.")
(PLOT/"supplementary/Figure_S14_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S15: command-permission effects.
d=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_PERMISSION_EFFECTS.csv")
d=d[(d.quality_scope=="primary")&(d.permission_step=="building_to_native_zone")&d.layout.isin(["random_mixing","demand_ordered"])].copy()
metrics=[("delta_mean_reference_mismatch_c",-1,"Mean mismatch reduction (°C)"),("delta_fixed_tail_reference_mismatch_c",-1,"Fixed-tail mismatch reduction (°C)"),("delta_modeled_interval_person_hour_coverage",100,"Interval coverage gain (percentage points)"),("delta_annual_site_kwh",1/1000,"Annual site-energy change (MWh)")]
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)
rng=np.random.default_rng(20261029)
for i,(col,mult,ylabel) in enumerate(metrics):
    ax=axes.flat[i]
    for j,cntl in enumerate(CONTROL_ORDER):
        vals=d[d.control==cntl][col].dropna().to_numpy()*mult
        ax.scatter(j+rng.normal(0,.045,len(vals)),vals,s=20,color=CONTROL_COLORS[cntl],alpha=.68,edgecolor="white",linewidth=.3)
        if len(vals):ax.plot([j-.15,j+.15],[np.median(vals)]*2,color="black",lw=1.4)
    ax.axhline(0,color="#B5B5B5",lw=.8);ax.set_xticks(range(3),["Uniform\naverage","Personal\ncentre","Personal\nrange"]);ax.set_ylabel(ylabel)
    final_title(ax,chr(97+i),["Population mean","Fixed reference tail","Personal range coverage","Energy consequence"][i]);final_style(ax)
save_plot(fig,"Figure_S15",size_mm=(180,128))
caption=("Supplementary Figure S15 | Effects of expanding command permission from one building-wide command to native-zone commands in matched layouts available to both archives. "
         "a,b, Reductions in population-average and fixed-reference-tail mismatch. c, Gain in occupied person-hour Personal range coverage. d, Change in annual site energy. Points are matched permission comparisons and black lines are medians; positive values in a–c denote service improvement.")
(PLOT/"supplementary/Figure_S15_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S16: common-baseline flexibility and information.
d=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_COMMON_BASELINE_DR.csv")
d=d[(d.quality_scope=="primary")&d.layout.isin(["random_mixing","demand_ordered"])].copy()
levels=["building_mean","zone_mean","full_distribution"]; labels=["Building\nmean","Zone\nmean","Full\ndistribution"]
condition_keys=["data_source","context","composition_id","layout","permission","event_month"]
ideal=d[d.information_level=="building_mean"].drop_duplicates(condition_keys)
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)
selected=[]; usable=[]
for level in levels:
    if level == "full_distribution":
        selected.append(int((ideal.ideal_full_information_response_kw>0).sum()))
        usable.append(int((ideal.ideal_full_information_response_kw>0).sum()))
    else:
        q=d[d.information_level==level]
        selected.append(int((q.selected_response_kw>0).sum()))
        usable.append(int((q.full_reviewed_usable_response_kw>0).sum()))
x=np.arange(3);w=.34
axes[0,0].bar(x-w/2,selected,w,color="#C6A064",label="Selected positive")
axes[0,0].bar(x+w/2,usable,w,color="#78A28E",label="Usable after full review")
axes[0,0].set_xticks(x,labels);axes[0,0].set_ylabel("Conditions");axes[0,0].legend(frameon=False,fontsize=9)
final_title(axes[0,0],"a","Information and usable response");final_style(axes[0,0])

false=[]
for permission in ["building","native_zone"]:
    false.append([int(((d.information_level==level)&(d.permission==permission)&(d.selected_response_kw>0)&(~d.selected_passes_full_review)).sum()) for level in levels[:2]])
im=axes[0,1].imshow(false,cmap="OrRd",aspect="auto",vmin=0)
for i in range(2):
    for j in range(2):axes[0,1].text(j,i,str(false[i][j]),ha="center",va="center")
axes[0,1].set_xticks([0,1],["Building mean","Zone mean"]);axes[0,1].set_yticks([0,1],["Building command","Native-zone commands"])
final_title(axes[0,1],"b","Mean-informed\nfalse acceptance");final_style(axes[0,1])

for j,permission in enumerate(["building","native_zone"]):
    vals=ideal[ideal.permission==permission].ideal_full_information_response_kw
    axes[1,0].hist(vals[vals>0],bins=12,histtype="step",lw=2,color=["#858684","#78A28E"][j],label=["Building command","Native-zone commands"][j])
axes[1,0].set_xlabel("Full-information response (kW)");axes[1,0].set_ylabel("Conditions");axes[1,0].legend(frameon=False,fontsize=9)
final_title(axes[1,0],"c","Qualified response distribution");final_style(axes[1,0])

for j,level in enumerate(levels[:2]):
    q=d[(d.information_level==level)&(d.selected_response_kw>0)].copy(); gap=q.selected_response_kw-q.full_reviewed_usable_response_kw
    axes[1,1].boxplot(gap,positions=[j],widths=.55,patch_artist=True,showfliers=False,boxprops=dict(facecolor=["#C98F7A","#9BBFC9"][j],edgecolor="#777777"),medianprops=dict(color="black"),whiskerprops=dict(color="#777777"),capprops=dict(color="#777777"))
axes[1,1].set_xticks([0,1],["Building mean","Zone mean"]);axes[1,1].set_ylabel("Selected minus usable response (kW)")
final_title(axes[1,1],"d","Response gap");final_style(axes[1,1])
save_plot(fig,"Figure_S16",size_mm=(180,128))
caption=("Supplementary Figure S16 | Common-baseline demand-response audit. All information levels use the same common physical baseline, action library, event windows and recovery review. "
         "a, Conditions in which each information level selected positive response and conditions retaining positive response after full-demand review. b, Positive mean-informed selections that did not pass full-demand review, separated by command permission. c, Positive response qualified using the full demand distribution. d, Difference between selected and full-reviewed usable response for mean-informed positive selections.")
(PLOT/"supplementary/Figure_S16_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S17: controller-own-baseline flexibility.
d=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_OWN_BASELINE_DR.csv")
d=d[(d.quality_scope=="primary")&d.layout.isin(["random_mixing","demand_ordered"])].copy()
c=pd.read_csv(RESULTS/"final_submission/evidence/FINAL_OWN_BASELINE_DR_COMPARISONS.csv")
c=c[(c.quality_scope=="primary")&c.layout.isin(["random_mixing","demand_ordered"])].copy()
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)
counts=[int((d[d.control==x].qualified_positive_response==True).sum()) for x in CONTROL_ORDER]
axes[0,0].bar(range(3),counts,color=[CONTROL_COLORS[x] for x in CONTROL_ORDER],width=.68)
for i,v in enumerate(counts):axes[0,0].text(i,v+1,str(v),ha="center")
axes[0,0].set_xticks(range(3),["Uniform\naverage","Personal\ncentre","Personal\nrange"]);axes[0,0].set_ylabel("Qualified positive conditions")
final_title(axes[0,0],"a","Response availability");final_style(axes[0,0])

for j,cntl in enumerate(CONTROL_ORDER):
    vals=d[(d.control==cntl)&(d.qualified_positive_response==True)].selected_response_kw.dropna()
    axes[0,1].boxplot(vals,positions=[j],widths=.55,patch_artist=True,showfliers=False,boxprops=dict(facecolor=CONTROL_COLORS[cntl],edgecolor="#777777"),medianprops=dict(color="black"),whiskerprops=dict(color="#777777"),capprops=dict(color="#777777"))
axes[0,1].set_xticks(range(3),["Uniform\naverage","Personal\ncentre","Personal\nrange"]);axes[0,1].set_ylabel("Qualified response (kW)")
final_title(axes[0,1],"b","Qualified response magnitude");final_style(axes[0,1])

for cntl in ["personal_center","personal_range"]:
    q=c[(c.control==cntl)&(c.uniform_response_kw>0)&(c.demand_guided_response_kw>0)]
    axes[1,0].scatter(q.uniform_response_kw,q.demand_guided_response_kw,s=25,color=CONTROL_COLORS[cntl],alpha=.72,label=CONTROL_LABELS[cntl],edgecolor="white",linewidth=.35)
mx=max(c.uniform_response_kw.max(),c.demand_guided_response_kw.max());axes[1,0].plot([0,mx],[0,mx],ls="--",color="#777777",lw=.8)
axes[1,0].set_xlabel("Uniform-average response (kW)");axes[1,0].set_ylabel("Demand-guided response (kW)");axes[1,0].legend(frameon=False,fontsize=9)
final_title(axes[1,0],"c","Paired own-baseline response");final_style(axes[1,0])

tab=d[d.qualified_positive_response==True].groupby(["event_month","data_source","control"]).size().rename("n").reset_index()
for j,month in enumerate([2,7]):
    q=tab[tab.event_month==month].groupby("control").n.sum().reindex(CONTROL_ORDER,fill_value=0)
    axes[1,1].bar(np.arange(3)+(j-.5)*.32,q,.32,color=["#9BBFC9","#C98F7A"][j],label=["February","July"][j])
axes[1,1].set_xticks(range(3),["Uniform\naverage","Personal\ncentre","Personal\nrange"]);axes[1,1].set_ylabel("Qualified positive conditions");axes[1,1].legend(frameon=False)
final_title(axes[1,1],"d","Event month");final_style(axes[1,1])
save_plot(fig,"Figure_S17",size_mm=(180,128))
caption=("Supplementary Figure S17 | Demand response measured from each controller's own annual operating baseline. a, Number of qualified positive response conditions by control. b, Magnitude of qualified positive response. "
         "c, Paired response when Uniform average and a demand-guided control both retained positive response; the dashed line is equality. d, Qualified positive conditions in February and July. Each response passes event and 48-hour recovery service checks.")
(PLOT/"supplementary/Figure_S17_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S18: event and recovery energetics across two accounting horizons.
d=pd.read_csv(RESULTS/'final_submission/evidence/FINAL_OWN_BASELINE_DR.csv')
d=d[(d.quality_scope=='primary')&d.layout.isin(['random_mixing','demand_ordered'])&(d.qualified_positive_response==True)].copy()
long=pd.read_csv(RESULTS/'final_submission/evidence/LONG_RECOVERY_TERMINAL_AUDIT.csv')
merge_keys=['data_source','context','composition_id','layout','permission','control','event_month','event_run_id','baseline_run_id']
d=d.merge(long[merge_keys+['net_electric_saving_168h_kwh','operational_co2_change_168h_kg','terminal_168h_max_abs_zone_temperature_delta_c']],on=merge_keys,how='left',validate='one_to_one')
d['recovery_ratio']=d.recovery_extra_electric_kwh/d.event_electric_saving_kwh
fig,axes=plt.subplots(3,2,figsize=(180/25.4,188/25.4));plt.subplots_adjust(hspace=.69,wspace=.40)
for cntl in CONTROL_ORDER:
    q=d[d.control==cntl]
    axes[0,0].scatter(q.event_electric_saving_kwh,q.recovery_extra_electric_kwh,s=25,color=CONTROL_COLORS[cntl],alpha=.72,label=CONTROL_LABELS[cntl],edgecolor='white',linewidth=.35)
mx=max(d.event_electric_saving_kwh.max(),d.recovery_extra_electric_kwh.max());axes[0,0].plot([0,mx],[0,mx],ls='--',color='#777777',lw=.8)
axes[0,0].set_xlabel('Event electricity saving (kWh)');axes[0,0].set_ylabel('48-h recovery electricity (kWh)',labelpad=1);axes[0,0].legend(frameon=False,fontsize=8.5)
final_title(axes[0,0],'a','Event saving and recovery');final_style(axes[0,0])
for j,cntl in enumerate(CONTROL_ORDER):
    vals=d[d.control==cntl].recovery_ratio.dropna()*100
    axes[0,1].boxplot(vals,positions=[j],widths=.55,patch_artist=True,showfliers=False,boxprops=dict(facecolor=CONTROL_COLORS[cntl],edgecolor='#777777'),medianprops=dict(color='black'),whiskerprops=dict(color='#777777'),capprops=dict(color='#777777'))
axes[0,1].axhline(100,color='#777777',ls='--',lw=.8);axes[0,1].set_xticks(range(3),['Uniform\naverage','Personal\ncentre','Personal\nrange']);axes[0,1].set_ylabel('Recovery/event electricity (%)',labelpad=1)
final_title(axes[0,1],'b','Recovery fraction');final_style(axes[0,1])
for j,cntl in enumerate(CONTROL_ORDER):
    vals=d[d.control==cntl].event_plus_recovery_net_electric_saving_kwh.dropna()
    axes[1,0].boxplot(vals,positions=[j],widths=.55,patch_artist=True,showfliers=False,boxprops=dict(facecolor=CONTROL_COLORS[cntl],edgecolor='#777777'),medianprops=dict(color='black'),whiskerprops=dict(color='#777777'),capprops=dict(color='#777777'))
axes[1,0].axhline(0,color='#B5B5B5',lw=.8);axes[1,0].set_xticks(range(3),['Uniform\naverage','Personal\ncentre','Personal\nrange']);axes[1,0].set_ylabel('Net electricity saving at 50 h (kWh)',labelpad=1)
final_title(axes[1,0],'c','Net electricity after 48-h recovery');final_style(axes[1,0])
for cntl in CONTROL_ORDER:
    q=d[d.control==cntl]
    axes[1,1].scatter(q.event_plus_recovery_net_electric_saving_kwh,q.event_plus_recovery_operational_co2_change_kg,s=25,color=CONTROL_COLORS[cntl],alpha=.72,edgecolor='white',linewidth=.35)
axes[1,1].axvline(0,color='#B5B5B5',lw=.8);axes[1,1].axhline(0,color='#B5B5B5',lw=.8);axes[1,1].set_xlabel('Net electricity saving at 50 h (kWh)');axes[1,1].set_ylabel('Operational carbon change (kgCO₂)',labelpad=1)
final_title(axes[1,1],'d','Net electricity and carbon');final_style(axes[1,1])
for cntl in CONTROL_ORDER:
    q=d[d.control==cntl]
    axes[2,0].scatter(q.event_plus_recovery_net_electric_saving_kwh,q.net_electric_saving_168h_kwh,s=25,color=CONTROL_COLORS[cntl],alpha=.72,edgecolor='white',linewidth=.35)
lo=min(d.event_plus_recovery_net_electric_saving_kwh.min(),d.net_electric_saving_168h_kwh.min());hi=max(d.event_plus_recovery_net_electric_saving_kwh.max(),d.net_electric_saving_168h_kwh.max());axes[2,0].plot([lo,hi],[lo,hi],ls='--',color='#777777',lw=.8);axes[2,0].axhline(0,color='#B5B5B5',lw=.8);axes[2,0].set_xlabel('Net saving at 50 h (kWh)');axes[2,0].set_ylabel('Net saving at 168 h (kWh)',labelpad=1)
final_title(axes[2,0],'e','Longer recovery horizon');final_style(axes[2,0]);axes[2,0].text(.98,.05,'96/97 remain positive',transform=axes[2,0].transAxes,ha='right',fontsize=9,fontweight='bold')
for j,cntl in enumerate(CONTROL_ORDER):
    vals=d[d.control==cntl].terminal_168h_max_abs_zone_temperature_delta_c.dropna()
    axes[2,1].boxplot(vals,positions=[j],widths=.55,patch_artist=True,showfliers=True,boxprops=dict(facecolor=CONTROL_COLORS[cntl],edgecolor='#777777'),medianprops=dict(color='black'),whiskerprops=dict(color='#777777'),capprops=dict(color='#777777'),flierprops=dict(marker='o',markersize=3,markerfacecolor=CONTROL_COLORS[cntl],markeredgecolor='white'))
axes[2,1].set_xticks(range(3),['Uniform\naverage','Personal\ncentre','Personal\nrange']);axes[2,1].set_ylabel('Maximum zone-temperature\ndifference at 168 h (°C)',labelpad=1)
final_title(axes[2,1],'f','Terminal indoor state');final_style(axes[2,1])
save_plot(fig,'Figure_S18',size_mm=(180,188))
caption=('Supplementary Figure S18 | Event and recovery energetics for 97 qualified positive own-baseline responses in the two shared-layout designs. a, Event-period electricity saving and additional electricity during the following 48 hours; the dashed line is equality. b, Recovery electricity as a percentage of event-period saving. c, Net electricity saving from event start through 50 hours. d, Electricity saving and operational-carbon change over the same 50-hour interval. e, Net electricity saving from event start through 50 and 168 hours; the dashed line is equality. f, Maximum absolute zone-temperature difference between perturbed and baseline operation at the end of the 168-hour interval. Boxes show medians and interquartile ranges with whiskers at 1.5 interquartile ranges; panel f retains individual values outside the whiskers.')
(PLOT/'supplementary/Figure_S18_caption.txt').write_text(caption,encoding='utf-8')


In [ ]:

# Figure_S19: implementation-resource scenario inputs.
d=pd.read_csv(RESULTS/"final_submission/costs/implementation_cost_scenarios.csv")
scope=pd.read_csv(RESULTS/"final_submission/costs/command_scope_management_boundaries.csv",low_memory=False)
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)
components=[("setup_hours","Setup hours"),("annual_maintenance_hours","Maintenance hours/year"),("annual_software_usd","Software (US$/year)"),("added_interface_capital_usd","Interface capital (US$)")]
for j,(col,label) in enumerate(components):
    vals=d[col].dropna(); q=np.quantile(vals,[0,.25,.5,.75,1]); span=q[-1]-q[0]
    qnorm=np.zeros_like(q) if span==0 else (q-q[0])/span*100
    axes[0,0].plot(qnorm,np.repeat(j,5),"o-",color=["#9BBFC9","#78A28E","#C6A064","#C98F7A"][j],lw=1.3,ms=4)
    axes[0,0].text(102,j,f"{q[0]:g}–{q[-1]:g}",va="center",fontsize=9)
axes[0,0].set_xlim(0,125);axes[0,0].set_yticks(range(4),[x[1] for x in components]);axes[0,0].set_xlabel("Position within declared range (%)")
final_title(axes[0,0],"a","Implementation input ranges");final_style(axes[0,0])

kinds=list(d.kind.dropna().unique())
for j,kind in enumerate(kinds):
    vals=d[d.kind==kind].annual_resource_cost_usd.dropna()
    axes[0,1].scatter(np.repeat(j,len(vals))+np.linspace(-.08,.08,len(vals)),vals,s=20,color=["#9BBFC9","#C6A064","#78A28E","#C98F7A"][j%4],alpha=.7)
axes[0,1].set_xticks(range(len(kinds)),[x.replace("_"," ").title() for x in kinds],rotation=25,ha="right");axes[0,1].set_ylabel("Annual resource cost (US$/year)")
final_title(axes[0,1],"b","Declared cost scenarios");final_style(axes[0,1])

axes[1,0].scatter(d.respondent_hours_year,d.annual_resource_cost_usd,s=24,color="#9A94B0",alpha=.68,edgecolor="white",linewidth=.35)
axes[1,0].set_xlabel("Respondent time (hours/year)");axes[1,0].set_ylabel("Annual resource cost (US$/year)")
final_title(axes[1,0],"c","Information and resource inputs");final_style(axes[1,0])

q=scope[["scope","declared_independent_command_groups","extra_groups_over_uniform"]].drop_duplicates()
for j,sname in enumerate(["building","floor","native_zone"]):
    vals=q[q.scope==sname].extra_groups_over_uniform.dropna()
    axes[1,1].boxplot(vals,positions=[j],widths=.55,patch_artist=True,showfliers=False,boxprops=dict(facecolor=["#858684","#C6A064","#78A28E"][j],edgecolor="#777777"),medianprops=dict(color="black"),whiskerprops=dict(color="#777777"),capprops=dict(color="#777777"))
axes[1,1].set_xticks(range(3),["Building","Floor","Native zone"]);axes[1,1].set_ylabel("Extra command groups over uniform")
final_title(axes[1,1],"d","Independent command groups");final_style(axes[1,1])
save_plot(fig,"Figure_S19",size_mm=(180,128))
caption=("Supplementary Figure S19 | Inputs to the implementation-resource scenarios. a, Declared ranges for setup, annual maintenance, software and added interface capital; points show minimum, quartiles, median and maximum normalized within each component, and labels give the actual minimum–maximum. "
         "b, Annual resource-cost values by scenario family. c, Respondent time and annual resource cost. d, Extra independently managed command groups relative to one uniform building command. These panels describe scenario inputs used in the cost sensitivity analysis.")
(PLOT/"supplementary/Figure_S19_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S20: bill and break-even sensitivity.
b=pd.read_csv(RESULTS/"final_submission/costs/bill_implementation_boundaries.csv")
a=pd.read_csv(RESULTS/"final_submission/costs/all_signed_paired_costs.csv",low_memory=False)
s=pd.read_csv(RESULTS/"final_submission/costs/command_scope_management_boundaries.csv",low_memory=False)
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)
for j,cntl in enumerate(["personal_center","personal_range"]):
    vals=b[(b.control==cntl)&(b.comparison_mode=="direct")].delta_energy_charge_usd_year.dropna()
    axes[0,0].boxplot(vals,positions=[j],widths=.55,patch_artist=True,showfliers=False,boxprops=dict(facecolor=CONTROL_COLORS[cntl],edgecolor="#777777"),medianprops=dict(color="black"),whiskerprops=dict(color="#777777"),capprops=dict(color="#777777"))
axes[0,0].axhline(0,color="#B5B5B5",lw=.8);axes[0,0].set_xticks([0,1],["Personal centre","Personal range"]);axes[0,0].set_ylabel("Annual energy-charge change (US$/year)")
final_title(axes[0,0],"a","Energy-bill change");final_style(axes[0,0])

for j,cntl in enumerate(["personal_center","personal_range"]):
    vals=b[(b.control==cntl)&(b.comparison_mode=="direct")].signed_additional_implementation_allowance_usd_year.dropna()
    axes[0,1].hist(vals,bins=18,histtype="step",lw=2,color=CONTROL_COLORS[cntl],label=CONTROL_LABELS[cntl])
axes[0,1].axvline(0,color="#777777",lw=.8);axes[0,1].set_xlabel("Energy-funded allowance (US$/year)");axes[0,1].set_ylabel("Comparisons");axes[0,1].legend(frameon=False,fontsize=9)
final_title(axes[0,1],"b","Energy-funded allowance");final_style(axes[0,1])

named=a[~a.scenario_id.str.startswith("grid_",na=False)].copy()
if named.empty: named=a.groupby(["comparison_id","control"],as_index=False).first()
for j,cntl in enumerate(["personal_center","personal_range"]):
    vals=named[named.control==cntl].minimum_additional_annual_external_value_for_resource_breakeven_usd.dropna()
    axes[1,0].boxplot(vals,positions=[j],widths=.55,patch_artist=True,showfliers=False,boxprops=dict(facecolor=CONTROL_COLORS[cntl],edgecolor="#777777"),medianprops=dict(color="black"),whiskerprops=dict(color="#777777"),capprops=dict(color="#777777"))
axes[1,0].set_xticks([0,1],["Personal centre","Personal range"]);axes[1,0].set_ylabel("Break-even external value (US$/year)")
final_title(axes[1,0],"c","Resource break-even threshold");final_style(axes[1,0])

q=s[np.isfinite(s.signed_resource_manager_hour_allowance_per_extra_group_year)].copy()
for j,scope_name in enumerate(["floor","native_zone"]):
    vals=q[q.scope==scope_name].signed_resource_manager_hour_allowance_per_extra_group_year
    axes[1,1].boxplot(vals,positions=[j],widths=.55,patch_artist=True,showfliers=False,boxprops=dict(facecolor=["#C6A064","#78A28E"][j],edgecolor="#777777"),medianprops=dict(color="black"),whiskerprops=dict(color="#777777"),capprops=dict(color="#777777"))
axes[1,1].axhline(0,color="#B5B5B5",lw=.8);axes[1,1].set_xticks([0,1],["Floor","Native zone"]);axes[1,1].set_ylabel("Manager-hour allowance per extra group/year")
final_title(axes[1,1],"d","Management boundary");final_style(axes[1,1])
save_plot(fig,"Figure_S20",size_mm=(180,128))
caption=("Supplementary Figure S20 | Bill and implementation sensitivity. a, Annual energy-charge change for direct demand-guided comparisons. b, Additional annual implementation allowance supplied by energy-charge savings; positive values indicate an energy-funded allowance. "
         "c, Additional annual external value required for resource break-even under the declared cost scenarios. d, Manager-hour allowance per extra command group for floor and native-zone permissions. Boxes show medians and interquartile ranges with whiskers at 1.5 interquartile ranges.")
(PLOT/"supplementary/Figure_S20_caption.txt").write_text(caption,encoding="utf-8")


In [ ]:

# Figure_S21: reproducibility and evidence-provenance QA.
qa=json.loads((RESULTS/"final_submission/evidence/FINAL_EVIDENCE_QA.json").read_text(encoding="utf-8"))
paths={
    "ASHRAE primary":RESULTS/"Reviewer_repair/repaired_controls_ashrae_full_analysis/ANALYSIS_QA.json",
    "ASHRAE strict":RESULTS/"Reviewer_repair/repaired_controls_ashrae_strict_analysis/ANALYSIS_QA.json",
    "CTCD primary":RESULTS/"Reviewer_repair/repaired_controls_ctcd_full_analysis/ANALYSIS_QA.json",
    "CTCD strict":RESULTS/"Reviewer_repair/repaired_controls_ctcd_strict_analysis/ANALYSIS_QA.json",
}
q={k:json.loads(p.read_text(encoding="utf-8")) for k,p in paths.items()}
fig,axes=plt.subplots(2,2,figsize=(180/25.4,128/25.4),constrained_layout=True)

stages=["Logical","Unique","Primary","Strict"]
for j,source in enumerate(["ASHRAE","CTCD"]):
    p=q[f"{source} primary"]; st=q[f"{source} strict"]
    vals=[p["logical_candidates"],p["unique_physical_runs"],p["engineering_qualified_runs"],st["engineering_qualified_runs"]]
    axes[0,0].plot(range(4),vals,"o-",lw=2,color=["#7FAFC4","#C98F7A"][j],label=source)
    for x,y in enumerate(vals):axes[0,0].text(x,y+35,str(y),ha="center",fontsize=9)
axes[0,0].set_xticks(range(4),stages);axes[0,0].set_ylabel("Runs or logical candidates");axes[0,0].legend(frameon=False)
final_title(axes[0,0],"a","Simulation evidence chain");final_style(axes[0,0])

rows=qa["output_rows"]; names=[x.replace("FINAL_","").replace(".csv","").replace("_"," ").title() for x in rows]; vals=list(rows.values()); order=np.argsort(vals)
axes[0,1].barh(np.arange(len(vals)),np.asarray(vals)[order],color="#9BBFC9")
axes[0,1].set_yticks(np.arange(len(vals)),np.asarray(names)[order],fontsize=8.5);axes[0,1].set_xlabel("Rows in final evidence table")
final_title(axes[0,1],"b","Final evidence tables");final_style(axes[0,1])

x=np.arange(2);w=.34
direct=[[q[f"{s} primary"]["direct_matched_pairs"],q[f"{s} strict"]["direct_matched_pairs"]] for s in ["ASHRAE","CTCD"]]
for j,scope in enumerate(["Primary","Strict"]):axes[1,0].bar(x+(j-.5)*w,[v[j] for v in direct],w,color=["#78A28E","#C6A064"][j],label=scope)
axes[1,0].set_xticks(x,["ASHRAE","CTCD"]);axes[1,0].set_ylabel("Matched direct comparisons");axes[1,0].legend(frameon=False)
final_title(axes[1,0],"c","Matched direct evidence");final_style(axes[1,0])

checks=[("Direct",69),("Common limits",198),("Seasonal",69),("Own baseline",288)]
axes[1,1].bar(range(len(checks)),[x[1] for x in checks],color=["#78A28E","#9BBFC9","#C6A064","#9A94B0"])
axes[1,1].set_xticks(range(len(checks)),[x[0].replace(" ","\n") for x in checks], rotation=30, ha="right");axes[1,1].set_ylabel("Evaluated or matched conditions")
for i,(_,v) in enumerate(checks):axes[1,1].text(i,v+7,str(v),ha="center")
final_title(axes[1,1],"d","Final analysis support");final_style(axes[1,1])
save_plot(fig,"Figure_S21",size_mm=(180,128))
caption=("Supplementary Figure S21 | Reproducibility and evidence-provenance checks. a, Progression from logical candidates to unique physical simulations and the primary and strict engineering-qualified sets for each demand archive. "
         "b, Row counts in the nine final evidence tables assembled from the declared analysis outputs. c, Matched direct comparisons supported under primary and strict checks. d, Condition counts for the principal direct-service, common-annual-limit, seasonal and own-baseline response analyses.")
(PLOT/"supplementary/Figure_S21_caption.txt").write_text(caption,encoding="utf-8")
